# Can 294 employees tell ten models apart?

The results table in this repository ranks ten models on one held-out set of **294
employees, 47 of whom left**. The best row, a grid-searched logistic regression, scores
0.8946 accuracy. The XGBoost fitted on the VIF feature set scores 0.8810.

That gap is **four employees**.

This notebook exists to work out whether a ranking built on 47 events is a result or a
reading of noise. It does not re-open the modelling. It rebuilds exactly the same ten
models on exactly the same split, under `SEED = 1234`, so every number here lines up with
the number in the main notebook, and then asks eight questions of them.

**The data is fictional.** IBM generated these 1,470 employees for a Watson Analytics
demo. Nobody in the file resigned from anything. Everything below is a statement about
*the method*, and none of it is a finding about any real workforce. That sentence is
repeated in a few places on purpose, because a notebook full of p-values invites the
reader to forget it.

**And this is not an argument that the models are useless.** But the claim that they beat
the 0.8401 majority-class baseline needs the same scrutiny as everything else, and
section 1c applies it: eight of the ten post a higher accuracy, five reach nominal
significance, and after correction none of them survives. What does hold, and what
sections 4 and 5 measure, is that they *rank* employees well even where the thresholded
labels cannot be separated from a constant. The finding is narrower and more awkward:
they cannot be ordered *against each other* on this test set. A ranking and a result are
different objects, and the table presents one as the other.

### The eight questions

| | question | instrument |
|---|---|---|
| 1 | Do any two models actually differ? | McNemar's exact test, all 45 pairs, Holm corrected |
| 2 | How wide is any single number? | Paired percentile bootstrap, 5,000 draws |
| 3 | What did `random_state=1234` buy? | RepeatedStratifiedKFold, 5 folds by 10 repeats |
| 4 | Which curve should be quoted at 16% prevalence? | ROC against precision-recall |
| 5 | Does a score of 0.7 mean a 70% chance? | Reliability curve, Brier decomposition |
| 6 | What is best-of-800 worth? | Selection optimism, CV against test |
| 7 | Do the coefficients mean what they are read to mean? | Unit scales against permutation importance |
| 8 | Would more data fix it? | Learning curve with a variability band |

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 100)

from scipy import stats
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, RepeatedStratifiedKFold,
    cross_validate, learning_curve
)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, brier_score_loss, confusion_matrix)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from statsmodels.stats.outliers_influence import variance_inflation_factor
import xgboost

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import gridspec
import plotly.offline as py
import plotly.graph_objs as go
from plotly.subplots import make_subplots
py.init_notebook_mode(connected=True)

SEED = 1234
B_BOOT = 5000          # bootstrap resamples
N_REPEATS = 10         # repeats of 5-fold cross validation
PATH = os.path.dirname(__file__) if '__file__' in locals() else os.getcwd()
os.chdir(PATH)

def show(df, fmt='{:.4f}'):
    '''Compact printer. Keeps this notebook readable rather than exhaustive.'''
    print(df.to_string(index=False, float_format=lambda v: fmt.format(v)))

print(
    'seeds fixed: SEED=%d, bootstrap draws=%d, CV repeats=%d' % (SEED, B_BOOT, N_REPEATS)
)

seeds fixed: SEED=1234, bootstrap draws=5000, CV repeats=10


## Rebuilding the committed split

Everything below is worthless if it is measured on a different split from the one the
results table reports. So the pipeline is reproduced line for line: drop the constant
columns and `EmployeeNumber`, map the seven ordinal codes to labels, split 80/20 at
`random_state=1234`, engineer `Generation`, `First_job_ind`, `Job_hop_idx` and
`compa_ratio` **after** the split, one-hot encode with `drop_first=True`.

The `compa_ratio` lookup is built on the full frame, exactly as the main notebook builds
it. That is the mild leak the repository's limitations section already admits to. It is
reproduced rather than fixed, because the job here is to measure the committed pipeline,
not a better one.

The asserts below are the receipt: 1,176 training rows, 294 test rows, 47 test leavers,
59 encoded features, 47 surviving VIF and 55 surviving the correlation filter. If any of
those moved, nothing downstream would be comparable to the published table.

In [2]:
def load_prepared():
    df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
    constant_cols = [c for c in df.columns if df[c].nunique() == 1]
    df = df.drop(columns=constant_cols).drop(columns=['EmployeeNumber'])
    ordinal_maps = {
        'Education': {
            1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Phd'
        },
        'EnvironmentSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
        'JobInvolvement': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
        'JobSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
        'PerformanceRating': {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'},
        'RelationshipSatisfaction': {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
        'WorkLifeBalance': {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
    }
    for col, mapping in ordinal_maps.items():
        df[col] = df[col].map(mapping)
    return df


def create_generation_feature(age_val):
    if age_val < 37:
        return 'Millenials'
    if age_val < 54:
        return 'Generation X'
    if age_val < 73:
        return 'Boomers'
    return 'Silent'


def engineer(X, lookup):
    X = X.copy()
    X['Generation'] = X.Age.apply(create_generation_feature)
    X['First_job_ind'] = np.where(X['NumCompaniesWorked'] == 0, 1, 0)
    X['Job_hop_idx'] = np.where(X['NumCompaniesWorked'] == 0, 0.,
                                X['TotalWorkingYears'] / X['NumCompaniesWorked'])
    merged = (X.reset_index()
               .merge(lookup, on=['Department', 'JobRole', 'JobLevel'], how='left')
               .set_index('index'))
    merged['compa_ratio'] = merged['MonthlyIncome'] / merged['MedianIncome']
    merged = merged.drop(columns=['MedianIncome', 'Count'])
    return merged.drop(columns=['Age', 'TotalWorkingYears', 'NumCompaniesWorked'])


def encode(df, object_cols, numeric_cols):
    dummies = [pd.get_dummies(df[c], prefix=c, drop_first=True, dtype='uint8')
               for c in object_cols]
    return pd.concat([df.loc[:, numeric_cols]] + dummies, axis=1)


def build_split():
    data_df = load_prepared()
    X = data_df.loc[:, data_df.columns != 'Attrition']
    y = data_df.loc[:, 'Attrition'].map({'Yes': 1, 'No': 0})
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=0.80, random_state=SEED)
    lookup = (data_df.groupby(['Department', 'JobRole', 'JobLevel'])
                     .agg(
                         MedianIncome=('MonthlyIncome', 'median'), Count=('Age', 'count')
                     )
                     .reset_index())
    X_train, X_test = engineer(X_train, lookup), engineer(X_test, lookup)
    object_cols = [c for c in X_train.columns if pd.api.types.is_string_dtype(X_train[c])]
    numeric_cols = X_train.columns.difference(object_cols)
    X_train = encode(X_train, object_cols, numeric_cols)
    X_test = encode(X_test, object_cols, numeric_cols)
    assert list(X_train.columns) == list(X_test.columns)
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = build_split()
y_true = y_test.values
N_TEST, N_POS = len(y_true), int(y_true.sum())
PREVALENCE = y_true.mean()
BASELINE_ACC = 1 - PREVALENCE

assert (X_train.shape, X_test.shape) == ((1176, 59), (294, 59))
assert (int(y_train.sum()), N_POS) == (190, 47)
print('train %s   test %s' % (X_train.shape, X_test.shape))
print('test leavers %d of %d, prevalence %.4f' % (N_POS, N_TEST, PREVALENCE))
print('majority-class baseline accuracy %.4f' % BASELINE_ACC)

train (1176, 59)   test (294, 59)
test leavers 47 of 294, prevalence 0.1599
majority-class baseline accuracy 0.8401


In [3]:
def calculate_vif_(df, thresh=5.0):
    variables = list(range(df.shape[1]))
    dropped = True
    while dropped:
        dropped = False
        vif = [variance_inflation_factor(df.iloc[:, variables].values, ix)
               for ix in range(df.iloc[:, variables].shape[1])]
        maxloc = vif.index(max(vif))
        if max(vif) > thresh:
            del variables[maxloc]
            dropped = True
    return df.columns[variables]


def get_remain_columns_using_corr(predictors_df, thresh=0.8):
    corr_matrix = predictors_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > thresh)]
    return upper.columns.difference(to_drop)


vif_cols_ = calculate_vif_(X_train, thresh=7.)
corr_cols_ = get_remain_columns_using_corr(X_train, thresh=.8)
assert (len(vif_cols_), len(corr_cols_)) == (47, 55)

FEATURE_SETS = {
    'all 59': (X_train, X_test),
    'VIF 47': (X_train.loc[:, vif_cols_], X_test.loc[:, vif_cols_]),
    'corr 55': (X_train.loc[:, corr_cols_], X_test.loc[:, corr_cols_]),
}
print('feature sets: ' + ', '.join('%s (%d cols)' % (k, v[0].shape[1])
                                   for k, v in FEATURE_SETS.items()))

feature sets: all 59 (59 cols), VIF 47 (47 cols), corr 55 (55 cols)


## The ten models, refitted

Two searches run first because three of the ten models are defined by their outcome: the
5-fold F1 grid search over `C`, `l1_ratio` and `class_weight` for the logistic
regressions, and the 800-candidate randomized search for the tuned XGBoost. Both are
re-run rather than pasted in, so that section 6 can look inside them.

The table this produces should match the repository's results table cell for cell. It is
printed here as the anchor for everything after it.

In [4]:
LOG_GRID = {'C': [0.001, 0.01, 0.1, 1., 10., 100.],
            'l1_ratio': [1.0, 0.0],
            'class_weight': [None, 'balanced']}
LOG_FIXED = {'random_state': SEED, 'solver': 'liblinear'}

log_search, log_best = {}, {}
for name, (A, _) in FEATURE_SETS.items():
    gs = GridSearchCV(LogisticRegression(**LOG_FIXED), LOG_GRID, scoring='f1', cv=5)
    gs.fit(A, y_train)
    log_search[name], log_best[name] = gs, gs.best_params_

XGB_GRID = {'n_estimators': [50, 100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'min_child_weight': [1, 2, 3, 5, 10], 'gamma': [0.1, 0.2, 0.3, 0.4, 0.5, 1],
            'subsample': [0.6, 0.7, 0.8], 'colsample_bytree': [0.6, 0.7, 0.8],
            'max_depth': [3, 4, 5]}
N_CANDIDATES = 800
xgb_search = RandomizedSearchCV(xgboost.XGBClassifier(random_state=SEED, n_jobs=-1),
                                param_distributions=XGB_GRID, n_iter=N_CANDIDATES,
                                scoring='f1', n_jobs=-1, cv=5, verbose=0,
                                random_state=SEED)
xgb_search.fit(X_train, y_train)

print('logistic grid winners (mean CV F1 of 24 candidates):')
for k, gs in log_search.items():
    print('  %-8s %s  cv_f1=%.4f' % (k, gs.best_params_, gs.best_score_))
print('randomized search winner (mean CV F1 of %d candidates): %.4f' %
      (N_CANDIDATES, xgb_search.best_score_))
print('  %s' % xgb_search.best_params_)

logistic grid winners (mean CV F1 of 24 candidates):
  all 59   {'C': 100.0, 'class_weight': None, 'l1_ratio': 1.0}  cv_f1=0.5224
  VIF 47   {'C': 100.0, 'class_weight': None, 'l1_ratio': 1.0}  cv_f1=0.4995
  corr 55  {'C': 100.0, 'class_weight': None, 'l1_ratio': 1.0}  cv_f1=0.5023
randomized search winner (mean CV F1 of 800 candidates): 0.4918
  {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.6}


In [5]:
def xgb_plain():
    return xgboost.XGBClassifier(random_state=SEED, n_jobs=-1, learning_rate=0.1,
                                 max_depth=3, n_estimators=100)

MODELS = [
    (
        'Decision tree',              'all 59',
        tree.DecisionTreeClassifier(random_state=SEED, max_depth=3)
    ),
    (
        'Random forest',              'all 59',
        RandomForestClassifier(n_estimators=100, max_depth=3, random_state=SEED)
    ),
    ('XGBoost',                    'all 59', xgb_plain()),
    ('XGBoost',                    'VIF 47', xgb_plain()),
    ('XGBoost',                    'corr 55', xgb_plain()),
    (
        'XGBoost, randomized search', 'all 59',
        xgboost.XGBClassifier(random_state=SEED, n_jobs=-1, **xgb_search.best_params_)
    ),
    ('Logistic regression',        'all 59', LogisticRegression(**LOG_FIXED)),
    (
        'Logistic regression, grid',  'all 59',
        LogisticRegression(**{**LOG_FIXED, **log_best['all 59']})
    ),
    (
        'Logistic regression, grid',  'VIF 47',
        LogisticRegression(**{**LOG_FIXED, **log_best['VIF 47']})
    ),
    (
        'Logistic regression, grid',  'corr 55',
        LogisticRegression(**{**LOG_FIXED, **log_best['corr 55']})
    ),
]
LABELS = ['%s [%s]' % (n, f) for n, f, _ in MODELS]
HEADLINE = 'Logistic regression, grid [all 59]'
RIVAL = 'XGBoost [VIF 47]'

fitted, y_hat, y_score = {}, {}, {}
rows = []
for (name, fs, est), label in zip(MODELS, LABELS):
    A, Bm = FEATURE_SETS[fs]
    est.fit(A, y_train)
    yp = est.predict(Bm)
    ps = est.predict_proba(Bm)[:, 1]
    fitted[label], y_hat[label], y_score[label] = est, yp, ps
    rows.append({'model': name, 'features': fs,
                 'accuracy': accuracy_score(y_true, yp),
                 'precision': precision_score(y_true, yp, zero_division=np.nan),
                 'recall': recall_score(y_true, yp),
                 'F1': f1_score(y_true, yp),
                 'ROC-AUC': roc_auc_score(y_true, ps),
                 'avg precision': average_precision_score(y_true, ps),
                 'errors': int((yp != y_true).sum())})
results = pd.DataFrame(rows, index=LABELS)

print(
    'Ten models on the same 294 employees. Baseline accuracy %.4f '
    '(predict nobody leaves).\n' % BASELINE_ACC
)
show(results)
print('\nheadline %s: %.4f accuracy, %d errors' %
      (HEADLINE, results.loc[HEADLINE, 'accuracy'], results.loc[HEADLINE, 'errors']))
print('rival    %s: %.4f accuracy, %d errors' %
      (RIVAL, results.loc[RIVAL, 'accuracy'], results.loc[RIVAL, 'errors']))
print('the entire headline gap is %d employees out of %d'
      % (results.loc[RIVAL, 'errors'] - results.loc[HEADLINE, 'errors'], N_TEST))
tn, fp, fn, tp = confusion_matrix(y_true, y_hat[HEADLINE]).ravel()
print('\nheadline confusion matrix (TN, FP, FN, TP): %d, %d, %d, %d' % (tn, fp, fn, tp))
print('it flags %d of the %d employees, %d of whom really left, and misses %d'
      % (tp + fp, N_TEST, tp, fn))

Ten models on the same 294 employees. Baseline accuracy 0.8401 (predict nobody leaves).

                     model features  accuracy  precision  recall     F1  ROC-AUC  avg precision  errors
             Decision tree   all 59    0.8401     0.5000  0.1064 0.1754   0.7639         0.3785      47
             Random forest   all 59    0.8401        NaN  0.0000 0.0000   0.8524         0.6074      47
                   XGBoost   all 59    0.8776     0.7619  0.3404 0.4706   0.8170         0.5870      36
                   XGBoost   VIF 47    0.8810     0.7727  0.3617 0.4928   0.8337         0.6451      35
                   XGBoost  corr 55    0.8741     0.7778  0.2979 0.4308   0.8198         0.5967      37
XGBoost, randomized search   all 59    0.8503     0.5455  0.3830 0.4500   0.8159         0.5864      44
       Logistic regression   all 59    0.8776     0.7037  0.4043 0.5135   0.8570         0.6179      36
 Logistic regression, grid   all 59    0.8946     0.7105  0.5745 0.6353   0.862

---

## 1. McNemar's exact test: do any two of these models actually differ?

This repository currently runs no significance test at all, and if it ran the obvious one
it would run the wrong one. The ten models scored **the same 294 employees**. That makes
the predictions *paired*: for each employee there are two verdicts, and the interesting
quantity is not how often each model is right but how often they disagree, and in whose
favour.

An unpaired comparison of two accuracies throws that structure away and treats 588
independent observations where there are 294. McNemar's test keeps it. It looks only at
the discordant pairs:

* **b** = employees model A got right and model B got wrong
* **c** = employees model B got right and model A got wrong

Under the null hypothesis that the two models are equally good, each discordant pair is a
coin flip, so `b ~ Binomial(b + c, 0.5)`. The exact binomial form is used below rather
than the chi-square approximation, because with 47 events the discordant counts run to a
couple of dozen and the approximation is not trustworthy there.

**Forty-five pairs are tested at once**, so a correction is not optional. At 5% per test,
the expected number of false positives across 45 independent tests is 2.25 even if every
model is identical. Holm's step-down procedure is used: sort the p-values ascending, and
compare the k-th smallest against `alpha / (m - k + 1)`, which in adjusted form multiplies
the k-th p-value by `(m - k + 1)` and enforces monotonicity. Holm controls the
family-wise error rate under any dependence structure, which matters here because these
45 tests are heavily dependent, sharing both models and employees.

In [6]:
def mcnemar_exact(correct_a, correct_b):
    '''Exact McNemar for two boolean vectors of per-employee correctness.'''
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    p = stats.binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else 1.0
    return b, c, p


def holm(pvals):
    '''Holm step-down adjusted p-values. Five lines, so it can be checked by
    reading it.'''
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


correct = {lab: (y_hat[lab] == y_true) for lab in LABELS}
pairs = []
for i in range(len(LABELS)):
    for j in range(i + 1, len(LABELS)):
        a, b_lab = LABELS[i], LABELS[j]
        b, c, p = mcnemar_exact(correct[a], correct[b_lab])
        pairs.append({'model A': a, 'model B': b_lab, 'b': b, 'c': c,
                      'discordant': b + c, 'net employees': b - c,
                      'acc diff': (b - c) / N_TEST, 'p': p})
mcnemar_df = pd.DataFrame(pairs)
mcnemar_df['p (Holm)'] = holm(mcnemar_df['p'].values)

n_raw = int((mcnemar_df['p'] < 0.05).sum())
n_holm = int((mcnemar_df['p (Holm)'] < 0.05).sum())
print('%d pairwise comparisons of %d employees each' % (len(mcnemar_df), N_TEST))
print('significant at raw p < 0.05:      %2d' % n_raw)
print('significant after Holm p < 0.05:  %2d' % n_holm)
print('smallest Holm-adjusted p-value:   %.4f' % mcnemar_df['p (Holm)'].min())
print('\nThe five smallest raw p-values, and what Holm does to them:')
show(mcnemar_df.nsmallest(5, 'p')[['model A', 'model B', 'b', 'c', 'discordant',
                                   'net employees', 'p', 'p (Holm)']])

WEAKEST = ['Decision tree [all 59]', 'Random forest [all 59]',
           'XGBoost, randomized search [all 59]']
nominal = mcnemar_df[mcnemar_df['p'] < 0.05]
involves_weak = nominal.apply(
    lambda r: r['model A'] in WEAKEST or r['model B'] in WEAKEST, axis=1
)
print(
    '\nOf the %d pairs reaching nominal significance, %d involve one of the three weakest'
    % (len(nominal), int(involves_weak.sum()))
)
print(
    'rows (decision tree, random forest, tuned XGBoost). Pairs among the other seven that'
)
print('reach even nominal significance: %d.' % int((~involves_weak).sum()))

45 pairwise comparisons of 294 employees each
significant at raw p < 0.05:      14
significant after Holm p < 0.05:   0
smallest Holm-adjusted p-value:   0.3323

The five smallest raw p-values, and what Holm does to them:
               model A                             model B  b  c  discordant  net employees      p  p (Holm)
Decision tree [all 59]                    XGBoost [all 59]  2 13          15            -11 0.0074    0.3323
Decision tree [all 59]                    XGBoost [VIF 47]  3 15          18            -12 0.0075    0.3323
Decision tree [all 59] Logistic regression, grid [corr 55]  9 25          34            -16 0.0090    0.3888
Decision tree [all 59]  Logistic regression, grid [all 59] 10 26          36            -16 0.0113    0.4759
Random forest [all 59] Logistic regression, grid [corr 55] 10 26          36            -16 0.0113    0.4759

Of the 14 pairs reaching nominal significance, 14 involve one of the three weakest
rows (decision tree, random forest, tune

In [7]:
head_pairs = mcnemar_df[
    (mcnemar_df['model A'] == HEADLINE) | (mcnemar_df['model B'] == HEADLINE)
].copy()
head_pairs['other'] = np.where(head_pairs['model A'] == HEADLINE,
                               head_pairs['model B'], head_pairs['model A'])
head_pairs['headline better by'] = np.where(head_pairs['model A'] == HEADLINE,
                                            head_pairs['b'] - head_pairs['c'],
                                            head_pairs['c'] - head_pairs['b'])
print('The headline model against each of the other nine, in employees:\n')
show(head_pairs.sort_values('p')[[
    'other', 'discordant', 'headline better by', 'p', 'p (Holm)'
]])

row = mcnemar_df[
    ((mcnemar_df['model A'] == HEADLINE) & (mcnemar_df['model B'] == RIVAL)) |
    ((mcnemar_df['model A'] == RIVAL) & (mcnemar_df['model B'] == HEADLINE))
].iloc[0]
print(
    '\nThe comparison the results table is built on, %s against %s:' % (HEADLINE, RIVAL)
)
print('  they disagree about %d employees' % row['discordant'])
print('  of those, the headline model wins %d and loses %d'
      % (max(row['b'], row['c']), min(row['b'], row['c'])))
print('  net %d employees, which is %.4f of accuracy'
      % (abs(row['net employees']), abs(row['acc diff'])))
print('  exact binomial p = %.4f, Holm-adjusted p = %.4f' % (row['p'], row['p (Holm)']))

The headline model against each of the other nine, in employees:

                              other  discordant  headline better by      p  p (Holm)
             Decision tree [all 59]          36                  16 0.0113    0.4759
             Random forest [all 59]          38                  16 0.0139    0.5541
XGBoost, randomized search [all 59]          27                  13 0.0192    0.7088
 Logistic regression, grid [VIF 47]           9                   5 0.1797    1.0000
                  XGBoost [corr 55]          24                   6 0.3075    1.0000
       Logistic regression [all 59]          17                   5 0.3323    1.0000
                   XGBoost [all 59]          21                   5 0.3833    1.0000
                   XGBoost [VIF 47]          26                   4 0.5572    1.0000
Logistic regression, grid [corr 55]           6                   0 1.0000    1.0000

The comparison the results table is built on, Logistic regression, grid [all 59] ag

### Why a heatmap for this

Forty-five numbers arranged as a list is a list; arranged as a symmetric grid it is a
shape, and the shape is the argument. A p-value matrix has exactly the structure a
heatmap wants: two categorical axes with the same ordering, one continuous value per
cell, and a threshold that the colour scale can be broken at. The eye reads "is anything
dark" in a single pass, which is the actual question, and no ordering of a 45-row table
does that.

Colour is scaled from 0 to 1 in the p-value itself rather than in a log, so the visual
distance between 0.02 and 0.55 is honest rather than dramatised. Hover carries what the
cell cannot: the discordant counts b and c, the net employees, and both the raw and the
Holm-adjusted p-value. Cells reaching significance after correction would be marked with
a cross. Whether any are is the point.

In [8]:
P_RAW = pd.DataFrame(np.ones((len(LABELS), len(LABELS))), index=LABELS, columns=LABELS)
P_ADJ = P_RAW.copy()
HOVER = np.empty((len(LABELS), len(LABELS)), dtype=object)
for lab in LABELS:
    HOVER[LABELS.index(lab), LABELS.index(lab)] = '%s<br>identical to itself' % lab
for _, r in mcnemar_df.iterrows():
    i, j = LABELS.index(r['model A']), LABELS.index(r['model B'])
    P_RAW.iloc[i, j] = P_RAW.iloc[j, i] = r['p']
    P_ADJ.iloc[i, j] = P_ADJ.iloc[j, i] = r['p (Holm)']
    txt = ('<b>%s</b><br>vs <b>%s</b><br>disagree about %d employees'
           '<br>b = %d, c = %d, net %+d<br>accuracy gap %+.4f'
           '<br>exact p = %.4f<br>Holm p = %.4f'
           % (r['model A'], r['model B'], r['discordant'], r['b'], r['c'],
              r['net employees'], r['acc diff'], r['p'], r['p (Holm)']))
    HOVER[i, j] = txt
    HOVER[j, i] = txt

marks_x, marks_y = [], []
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j and P_ADJ.iloc[i, j] < 0.05:
            marks_x.append(LABELS[j]); marks_y.append(LABELS[i])

heat = go.Heatmap(z=P_RAW.values, x=LABELS, y=LABELS, zmin=0, zmax=1,
                  colorscale=[[0.0, '#7f0000'], [0.05, '#d7301f'], [0.0500001, '#fdd49e'],
                              [0.5, '#e0ecf4'], [1.0, '#4d648d']],
                  colorbar=dict(title='raw p', tickvals=[0, 0.05, 0.25, 0.5, 0.75, 1.0]),
                  text=HOVER, hoverinfo='text')
data = [heat]
if marks_x:
    data.append(go.Scatter(x=marks_x, y=marks_y, mode='markers',
                           marker=dict(symbol='x', size=12, color='black'),
                           name='significant after Holm', hoverinfo='skip'))
layout = go.Layout(
    title=('Pairwise McNemar exact p-values, 45 comparisons of the same 294 employees'
           '<br><sub>%d cells below raw 0.05, %d surviving Holm correction. '
           'Crosses would mark survivors.</sub>' % (n_raw * 2, n_holm * 2)),
    height=760, width=900, margin=dict(l=260, b=260, t=90, r=40),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9), constrain='domain'),
    yaxis=dict(tickfont=dict(size=9), autorange='reversed', scaleanchor='x'))
py.iplot(go.Figure(data=data, layout=layout))

### What the matrix says

**Nothing is dark after correction.** Fourteen of the 45 raw p-values fall below 0.05, and
every one of them is a comparison against the decision tree, the random forest, or the
tuned XGBoost, which are the three weakest rows in the table. Not one of those fourteen
survives Holm. The smallest adjusted p-value in the whole matrix is 0.33.

Read that carefully, because it is easy to over-claim in either direction. Failing to
reject is not proof of equality. It is a statement that **this test set is too small to
resolve the differences it is being asked to resolve**, which is a fact about the
measuring instrument rather than about the models. Section 2 puts a number on how small.

And the specific comparison the results table leads with, the headline logistic
regression against the VIF XGBoost, is not close to significance even before correction.
The two models disagree about 26 employees, and split them 15 to 11. A fair coin does
that about half the time.

---

## 2. Bootstrap intervals: how wide is any one of these numbers?

McNemar answers "do two models differ". It does not answer "what is the accuracy". For
that, resample.

The percentile bootstrap here draws 5,000 test sets of 294 employees, with replacement,
from the 294 that exist, and recomputes every metric on each draw. The 2.5th and 97.5th
percentiles of the resulting distribution give a 95% interval.

Two deliberate choices:

**The resample is paired across models.** All ten models are scored on the *same* 5,000
bootstrap index sets, not on ten independent sets. That costs nothing and buys the
ability to bootstrap the *difference* between two models, which is a much sharper
instrument than comparing two overlapping intervals. Overlapping intervals prove nothing
about a difference: two intervals can overlap while the paired difference is clearly
non-zero, and reading the gap between error bars answers a different and more
conservative question than the one being asked.

**The seed is fixed and reported: `numpy.random.default_rng(1234)`.** A bootstrap with an
unreported seed is a number nobody can check.

Precision is undefined on any draw where a model predicts no leavers at all. Those draws
are dropped from that model's precision interval and the share dropped is reported,
because a precision interval computed over an unknown subset of draws is not an interval.
For the random forest, which predicts zero leavers on the real test set, it is undefined
everywhere.

In [9]:
rng = np.random.default_rng(SEED)
BOOT_IDX = rng.integers(0, N_TEST, size=(B_BOOT, N_TEST))


def fast_auc(y, s):
    '''Rank-based ROC-AUC, tie-corrected. 5,000 draws x 10 models needs it to be quick.'''
    order = np.argsort(s, kind='stable')
    s_sorted, y_sorted = s[order], y[order]
    ranks = np.empty(len(s), dtype=float)
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s_sorted[j + 1] == s_sorted[i]:
            j += 1
        ranks[i:j + 1] = (i + j) / 2.0 + 1.0
        i = j + 1
    n_pos = int(y_sorted.sum())
    n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    return (ranks[y_sorted == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


boot = {}
y_boot = y_true[BOOT_IDX]
for lab in LABELS:
    yp = y_hat[lab][BOOT_IDX]
    ps = y_score[lab][BOOT_IDX]
    tp = ((yp == 1) & (y_boot == 1)).sum(1).astype(float)
    fp = ((yp == 1) & (y_boot == 0)).sum(1).astype(float)
    fn = ((yp == 0) & (y_boot == 1)).sum(1).astype(float)
    tn = ((yp == 0) & (y_boot == 0)).sum(1).astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        boot[lab] = {
            'accuracy':  (tp + tn) / N_TEST,
            'precision': np.where(tp + fp > 0, tp / (tp + fp), np.nan),
            'recall':    np.where(tp + fn > 0, tp / (tp + fn), np.nan),
            'F1':        np.where(
                2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan
            ),
            'ROC-AUC':   np.array([fast_auc(y_boot[k], ps[k]) for k in range(B_BOOT)]),
        }

METRICS = ['accuracy', 'precision', 'recall', 'F1', 'ROC-AUC']


def pct_ci(v, lo=2.5, hi=97.5):
    v = v[~np.isnan(v)]
    return (np.nan, np.nan) if len(v) == 0 else tuple(np.percentile(v, [lo, hi]))


ci_rows = []
for lab in LABELS:
    r = {'model': lab}
    for m in METRICS:
        lo, hi = pct_ci(boot[lab][m])
        r[m] = results.loc[lab, m]
        r[m + ' lo'], r[m + ' hi'], r[m + ' width'] = lo, hi, hi - lo
    r['precision undefined'] = float(np.isnan(boot[lab]['precision']).mean())
    ci_rows.append(r)
ci = pd.DataFrame(ci_rows).set_index('model')

print('95%% percentile bootstrap, %d draws, seed %d.\n' % (B_BOOT, SEED))
for m in ['accuracy', 'F1', 'ROC-AUC']:
    out = ci[[m, m + ' lo', m + ' hi', m + ' width']].copy()
    out.columns = [m, 'lo', 'hi', 'width']
    print(m.upper())
    print(out.to_string(float_format=lambda v: '%.4f' % v))
    print()
print('precision undefined in this share of draws:')
print(ci['precision undefined'][ci['precision undefined'] > 0].to_string(
    float_format=lambda v: '%.4f' % v))

95% percentile bootstrap, 5000 draws, seed 1234.

ACCURACY
                                     accuracy     lo     hi  width
model                                                             
Decision tree [all 59]                 0.8401 0.7959 0.8810 0.0850
Random forest [all 59]                 0.8401 0.7992 0.8810 0.0817
XGBoost [all 59]                       0.8776 0.8367 0.9116 0.0748
XGBoost [VIF 47]                       0.8810 0.8435 0.9150 0.0714
XGBoost [corr 55]                      0.8741 0.8333 0.9116 0.0782
XGBoost, randomized search [all 59]    0.8503 0.8061 0.8878 0.0816
Logistic regression [all 59]           0.8776 0.8401 0.9150 0.0748
Logistic regression, grid [all 59]     0.8946 0.8571 0.9286 0.0714
Logistic regression, grid [VIF 47]     0.8776 0.8367 0.9116 0.0748
Logistic regression, grid [corr 55]    0.8946 0.8571 0.9286 0.0714

F1
                                        F1     lo     hi  width
model                                                          
Decis

In [10]:
spread = results['accuracy'].max() - results['accuracy'].min()
widest = ci['accuracy width'].max()
narrowest = ci['accuracy width'].min()
print('The ten accuracy point estimates span            %.4f  (%.1f employees)'
      % (spread, spread * N_TEST))
print('The narrowest single 95%% interval is             %.4f wide  (%.1f employees)'
      % (narrowest, narrowest * N_TEST))
print('The widest single 95%% interval is                %.4f wide  (%.1f employees)'
      % (widest, widest * N_TEST))
print('\nEvery model in the table fits inside the error bar of any one row.'
      if spread < narrowest else '\nThe spread exceeds the narrowest interval.')
print('\nF1 tells the same story more loudly: the ten point estimates span %.4f,'
      % (results['F1'].max() - results['F1'].min()))
print(
    'and the narrowest F1 interval among the models that predict any leaver is %.4f wide.'
    % ci.loc[ci['F1'] > 0, 'F1 width'].min()
)

The ten accuracy point estimates span            0.0544  (16.0 employees)
The narrowest single 95% interval is             0.0714 wide  (21.0 employees)
The widest single 95% interval is                0.0850 wide  (25.0 employees)

Every model in the table fits inside the error bar of any one row.

F1 tells the same story more loudly: the ten point estimates span 0.6353,
and the narrowest F1 interval among the models that predict any leaver is 0.2450 wide.


### Why a forest plot

A forest plot is the right shape whenever the question is "are these estimates
distinguishable" rather than "which is biggest". It puts each estimate on its own row,
draws the interval as a horizontal line at the estimate's own height, and lets the reader
sweep a vertical eye-line down the page. Ten overlapping intervals stacked that way make
the argument in one glance, and no table of ten rows by three columns can, because a
table asks the reader to hold twenty numbers in their head and do the comparison
themselves.

Rows are ordered by point estimate, which is the ordering the results table implies is
meaningful. The majority-class baseline is drawn as a vertical reference line, because
accuracy read against zero is meaningless at 16% prevalence and every reader who has seen
"89% accurate" in a slide deck needs that line in front of them.

Hover carries the interval bounds, the width in employees, and the number of test-set
errors, none of which fit on the chart without ruining it.

In [11]:
order = results['accuracy'].sort_values().index.tolist()
lo = ci.loc[order, 'accuracy lo'].values
hi = ci.loc[order, 'accuracy hi'].values
pt = results.loc[order, 'accuracy'].values
err = results.loc[order, 'errors'].values

hover = [
    '<b>%s</b><br>accuracy %.4f<br>95%% CI [%.4f, %.4f]'
    '<br>interval width %.4f = %.0f employees<br>%d of %d test employees misclassified'
    % (lab, p, l, h, h - l, (h - l) * N_TEST, e, N_TEST)
    for lab, p, l, h, e in zip(order, pt, lo, hi, err)
]

traces = [go.Scatter(x=[l, h], y=[lab, lab], mode='lines',
                     line=dict(color='rgba(70,100,150,0.85)', width=3),
                     hoverinfo='skip', showlegend=False)
          for lab, l, h in zip(order, lo, hi)]
traces.append(go.Scatter(x=pt, y=order, mode='markers',
                         marker=dict(size=11, color='#16406e',
                                     line=dict(color='white', width=1.2)),
                         text=hover, hoverinfo='text', name='accuracy'))
traces.append(go.Scatter(x=[BASELINE_ACC, BASELINE_ACC], y=[order[0], order[-1]],
                         mode='lines', line=dict(color='#c0392b', width=2, dash='dash'),
                         name='predict nobody leaves (%.4f)' % BASELINE_ACC,
                         hoverinfo='name'))
layout = go.Layout(
    title=(
        'Ten models, ten 95%% bootstrap intervals, one test set of %d employees'
        '<br><sub>%d percentile bootstrap draws, seed %d. The ten point estimates span '
        '%.4f. The narrowest single interval is %.4f wide.</sub>'
        % (N_TEST, B_BOOT, SEED, spread, narrowest)
    ),
    height=560, width=980, margin=dict(l=280, r=40, t=110, b=60),
    xaxis=dict(title='accuracy on the held-out 294', zeroline=False,
               range=[min(lo) - 0.02, max(hi) + 0.02]),
    yaxis=dict(tickfont=dict(size=10)),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.7)'),
    hovermode='closest')
py.iplot(go.Figure(data=traces, layout=layout))

### The same picture across all five metrics

One metric can flatter. Five small multiples, sharing a model ordering so the eye can
track a row across panels, show whether any metric separates these models. Each panel
carries its own reference line, because "no skill" means a different number in each:
0.8401 for accuracy, the 0.1599 prevalence for precision, 0.5 for ROC-AUC. Recall and F1
have no single sensible null so none is drawn.

Note what the panels do to the ordering. The rows are sorted by F1, and the accuracy
panel is not monotone in that order. Two models tie at the top of accuracy while sitting
two rows apart on F1, and the random forest sits at the baseline on accuracy with an F1
of exactly zero and a respectable ROC-AUC. Which model is "best" depends on the column,
and the column was a choice.

In [12]:
order_f1 = results['F1'].sort_values().index.tolist()
NULLS = {'accuracy': BASELINE_ACC, 'precision': PREVALENCE, 'ROC-AUC': 0.5}
fig = make_subplots(rows=1, cols=5, shared_yaxes=True, horizontal_spacing=0.012,
                    subplot_titles=METRICS)
for k, m in enumerate(METRICS, start=1):
    l = ci.loc[order_f1, m + ' lo'].values
    h = ci.loc[order_f1, m + ' hi'].values
    p = results.loc[order_f1, m].values
    for lab, a, b_ in zip(order_f1, l, h):
        if np.isnan(a):
            continue
        fig.add_trace(go.Scatter(x=[a, b_], y=[lab, lab], mode='lines',
                                 line=dict(color='rgba(70,100,150,0.75)', width=2.5),
                                 hoverinfo='skip', showlegend=False), row=1, col=k)
    txt = ['<b>%s</b><br>%s %.4f<br>95%% CI [%.4f, %.4f]<br>width %.4f'
           % (lab, m, pp, ll, hh, hh - ll) if not np.isnan(ll)
           else '<b>%s</b><br>%s undefined' % (lab, m)
           for lab, pp, ll, hh in zip(order_f1, p, l, h)]
    fig.add_trace(go.Scatter(x=p, y=order_f1, mode='markers',
                             marker=dict(size=8, color='#16406e'),
                             text=txt, hoverinfo='text', showlegend=False), row=1, col=k)
    if m in NULLS:
        fig.add_trace(go.Scatter(
            x=[NULLS[m]] * 2, y=[order_f1[0], order_f1[-1]], mode='lines',
            line=dict(color='#c0392b', width=1.5, dash='dash'), hoverinfo='skip',
            showlegend=False
        ), row=1, col=k)
    fig.update_xaxes(range=[-0.03, 1.03], dtick=0.25, tickfont=dict(size=9), row=1, col=k)
fig.update_yaxes(tickfont=dict(size=10), row=1, col=1)
fig.update_layout(
    height=520, width=1050, margin=dict(l=270, r=25, t=100, b=50),
    title=('Every metric, every model, with its 95%% bootstrap interval'
           '<br><sub>Rows ordered by F1. Dashed line is the no-skill value for '
           'that metric where one exists. Precision is undefined for the random '
           'forest, which predicts no leaver.</sub>')
)
py.iplot(fig)

In [13]:
diffs = []
for lab in LABELS:
    if lab == HEADLINE:
        continue
    d = boot[HEADLINE]['accuracy'] - boot[lab]['accuracy']
    lo_d, hi_d = np.percentile(d, [2.5, 97.5])
    diffs.append({
        'against': lab,
        'observed gap': results.loc[HEADLINE, 'accuracy'] - results.loc[lab, 'accuracy'],
        'employees': int(results.loc[lab, 'errors'] - results.loc[HEADLINE, 'errors']),
        'CI lo': lo_d, 'CI hi': hi_d,
        'P(headline better)': float(np.mean(d > 0)),
        'CI excludes 0': bool(lo_d > 0 or hi_d < 0)
    })
diff_df = pd.DataFrame(diffs).sort_values('observed gap', ascending=False)
print('Paired bootstrap of the accuracy DIFFERENCE, headline model minus each rival.')
print(
    'This is the sharper instrument: it uses the same resampled employees for both '
    'models.\n'
)
show(diff_df)
print('\ndifferences whose 95%% interval excludes zero: %d of %d'
      % (int(diff_df['CI excludes 0'].sum()), len(diff_df)))
print('note these %d are uncorrected for multiplicity; McNemar with Holm found none.'
      % int(diff_df['CI excludes 0'].sum()))

Paired bootstrap of the accuracy DIFFERENCE, headline model minus each rival.
This is the sharper instrument: it uses the same resampled employees for both models.

                            against  observed gap  employees   CI lo  CI hi  P(headline better)  CI excludes 0
             Decision tree [all 59]        0.0544         16  0.0170 0.0952              0.9950           True
             Random forest [all 59]        0.0544         16  0.0136 0.0952              0.9960           True
XGBoost, randomized search [all 59]        0.0442         13  0.0102 0.0782              0.9962           True
                  XGBoost [corr 55]        0.0204          6 -0.0102 0.0544              0.8774          False
                   XGBoost [all 59]        0.0170          5 -0.0136 0.0476              0.8446          False
       Logistic regression [all 59]        0.0170          5 -0.0102 0.0442              0.8626          False
 Logistic regression, grid [VIF 47]        0.0170         

### How large would the test set have to be?

The honest follow-up to "not significant" is "then what would be enough", and it is
answerable. McNemar's power depends on two quantities that the current test set
estimates directly: the rate at which two models disagree at all, and how lopsidedly the
disagreements fall.

For a normal approximation to the binomial with `p` the probability that a discordant
pair favours the better model, the number of discordant pairs needed for a two-sided test
at `alpha` with power `1 - beta` is

```
n_disc = (z_alpha/2 * 0.5 + z_beta * sqrt(p(1-p)))^2 / (p - 0.5)^2
```

and dividing by the observed discordance rate converts that into employees. The closed
form is checked below against a Monte Carlo run of the exact test, because a formula that
has not been simulated against is a formula that might be wrong.

In [14]:
def mcnemar_min_n(b, c, n, alpha=0.05, power=0.80):
    disc = b + c
    if disc == 0 or b == c:
        return np.inf, np.inf
    disc_rate = disc / n
    p = max(b, c) / disc
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    n_disc = (z_a * 0.5 + z_b * np.sqrt(p * (1 - p))) ** 2 / (p - 0.5) ** 2
    return n_disc, n_disc / disc_rate


def simulate_power(b, c, n_employees, n_sim=2000, alpha=0.05, seed=SEED):
    '''Monte Carlo power of the exact test at a given test-set size, under the
    observed effect.'''
    disc = b + c
    r = np.random.default_rng(seed)
    p = max(b, c) / disc
    rate = disc / N_TEST
    n_disc = r.binomial(n_employees, rate, size=n_sim)
    wins = r.binomial(np.maximum(n_disc, 1), p)
    hits = 0
    for d, w in zip(n_disc, wins):
        if d > 0 and stats.binomtest(int(w), int(d), 0.5).pvalue < alpha:
            hits += 1
    return hits / n_sim


b_obs, c_obs = int(row['b']), int(row['c'])
need_disc, need_n = mcnemar_min_n(b_obs, c_obs, N_TEST)
print('Observed for %s vs %s: b=%d, c=%d out of %d employees'
      % (HEADLINE, RIVAL, b_obs, c_obs, N_TEST))
print(
    '  they disagree about %.2f%% of employees, split %.4f in the headline model\'s '
    'favour' % (100 * (b_obs + c_obs) / N_TEST, max(b_obs, c_obs) / (b_obs + c_obs))
)
print('  discordant pairs needed for 80%% power: %.0f' % need_disc)
print('  which at that disagreement rate is a test set of %.0f employees' % need_n)
print('  the whole dataset, train and test together, is 1,470')
print(
    '\nMonte Carlo check of the exact test under the observed effect '
    '(%d simulations each):' % 2000
)
for n_emp in [N_TEST, 1000, int(round(need_n)), 10000]:
    print(
        '  test set of %6d employees -> power %.3f'
        % (n_emp, simulate_power(b_obs, c_obs, n_emp))
    )

Observed for Logistic regression, grid [all 59] vs XGBoost [VIF 47]: b=11, c=15 out of 294 employees
  they disagree about 8.84% of employees, split 0.5769 in the headline model's favour
  discordant pairs needed for 80% power: 329
  which at that disagreement rate is a test set of 3723 employees
  the whole dataset, train and test together, is 1,470

Monte Carlo check of the exact test under the observed effect (2000 simulations each):
  test set of    294 employees -> power 0.084


  test set of   1000 employees -> power 0.268


  test set of   3723 employees -> power 0.769
  test set of  10000 employees -> power 0.992


### 1c. And against the baseline itself?

Sections 1 and 1b asked whether the models differ from each other. They do not. But the
intro made a second claim, that the models at least beat the majority-class baseline, and
a notebook about untested claims should not leave its own untested.

The baseline is the rule "predict nobody leaves". It is right about all 247 stayers and
wrong about all 47 leavers. So against it, McNemar's discordant pairs are exactly the
model's true positives and its false positives: a true positive is a leaver the model
found and the baseline missed, a false positive is a stayer the model disturbed and the
baseline left alone. The same instrument as before, pointed one row lower.


In [15]:
# Every model against the rule "predict nobody leaves". Discordant pairs against that
# baseline are exactly TP (model right, baseline wrong) and FP (baseline right, model
# wrong), so the same exact binomial applies, with the same Holm correction as section 1.
baseline_rows = []
for label in LABELS:
    cm = confusion_matrix(y_true, y_hat[label])
    tp, fp = int(cm[1, 1]), int(cm[0, 1])
    n = tp + fp
    p = 1.0 if n == 0 else float(stats.binomtest(min(tp, fp), n, 0.5).pvalue)
    baseline_rows.append({'model': label, 'found (TP)': tp, 'disturbed (FP)': fp,
                          'net employees': tp - fp, 'p': p})

baseline_df = pd.DataFrame(baseline_rows).sort_values('p').reset_index(drop=True)
running = 0.0
holm = []
for i, raw_p in enumerate(baseline_df['p']):
    running = max(running, (len(baseline_df) - i) * raw_p)
    holm.append(min(1.0, running))
baseline_df['p (Holm)'] = holm

print('Each model against the majority-class baseline (predict nobody leaves):\n')
print(baseline_df.to_string(index=False, float_format=lambda v: format(v, '.4f')))
print('\nreach p < 0.05 uncorrected : %d of %d'
      % ((baseline_df['p'] < 0.05).sum(), len(baseline_df)))
print('survive Holm correction    : %d of %d'
      % ((baseline_df['p (Holm)'] < 0.05).sum(), len(baseline_df)))
print('smallest Holm-adjusted p   : %.4f (%s)'
      % (baseline_df['p (Holm)'].min(), baseline_df.iloc[0]['model']))

Each model against the majority-class baseline (predict nobody leaves):

                              model  found (TP)  disturbed (FP)  net employees      p  p (Holm)
Logistic regression, grid [corr 55]          26              10             16 0.0113    0.1133
 Logistic regression, grid [all 59]          27              11             16 0.0139    0.1247
                   XGBoost [VIF 47]          17               5             12 0.0169    0.1352
                   XGBoost [all 59]          16               5             11 0.0266    0.1862
                  XGBoost [corr 55]          14               4             10 0.0309    0.1862
       Logistic regression [all 59]          19               8             11 0.0522    0.2612
 Logistic regression, grid [VIF 47]          22              11             11 0.0801    0.3206
XGBoost, randomized search [all 59]          18              15              3 0.7283    1.0000
             Decision tree [all 59]           5               5

The claim in the intro was too strong, and this is the test that shows it.

Eight of the ten models post a higher accuracy than 0.8401, which is what "beats the
baseline" usually means when someone says it out loud. Five of them reach nominal
significance. **After the same Holm correction applied to the pairwise comparisons in
section 1, not one of the ten survives.** The strongest result on the page, the
correlation-feature logistic regression, lands at an adjusted p of 0.11.

That is a harder finding than the one this notebook set out to report, and it deserves
saying rather than softening. On 47 events, this test set cannot separate the models from
each other, and it cannot separate any of them from a rule that ignores the data entirely.

Two things stop that from collapsing into nihilism, and both are measured further down.
Ranking is not classification: section 4 shows the headline model reaching an average
precision of 0.66 against a prevalence floor of 0.16, so the scores order employees far
better than chance even where the thresholded labels cannot be told from a constant. And
section 5 shows those scores are calibrated in aggregate, with the top decile holding 23
leavers out of 30. Sorting works. It is the accept-or-reject decision at 0.5 that this
test set is too small to defend.


The current test set has roughly a one-in-ten chance of detecting the very effect it is
being used to claim. To get to the conventional 80%, the held-out set would need to be
several times the size of the entire dataset, and this dataset has 1,470 rows in total.

That is the whole notebook in one number. The comparison is not merely unproven, it is
**unprovable at this sample size**, and no amount of re-running the split changes that.

Note carefully what this calculation does *not* claim. It is powered against the effect
size that was *observed*, and the observed effect is itself a noisy estimate drawn from
this same small sample. If the true difference were larger, less data would be needed. It
is a statement about what it would take to confirm the gap as measured, not a
pronouncement that the two models are truly identical.

---

## 3. Split sensitivity: what did `random_state=1234` buy?

Every number in the results table is conditional on one draw. `train_test_split` was
called once, with seed 1234, and the 294 employees it happened to select are the entire
evidential basis of the ranking. The repository's limitations section admits this. This
section measures it.

`RepeatedStratifiedKFold(n_splits=5, n_repeats=10)` produces 50 train/test partitions.
Five folds is the right choice here rather than ten, because 5-fold holds out 20% of the
data, which is exactly the proportion the committed split holds out. Each fold score is
therefore directly comparable to the committed number: same training size, same test
size, different employees.

Two honest caveats, and they both cut the same way:

* **The hyperparameters are frozen at their committed values.** The logistic grid and the
  800-candidate randomized search both chose their winners on the committed training set.
  Re-running the searches inside every fold would be the fully correct nested procedure.
  Not doing it means the cross-validated scores here are mildly optimistic, because
  hyperparameters selected using data now appearing in a fold's test partition leak into
  that fold. The same applies to the VIF and correlation feature sets, which were also
  selected on the committed training set.
* **The 50 fold scores are not 50 independent observations.** Folds within a repeat share
  training data, and repeats reuse all 1,470 rows. A standard error computed as
  `sd / sqrt(50)` would be too small, which is the well-known problem with naive
  cross-validation variance estimates. So no such standard error is quoted below. The
  *spread* is what is being reported, and the spread is honest.

In [16]:
X_all = pd.concat([X_train, X_test]).sort_index()
y_all = pd.concat([y_train, y_test]).sort_index()
assert X_all.shape == (1470, 59) and int(y_all.sum()) == 237

CV_SPECS = [
    (
        'Decision tree [all 59]',
        tree.DecisionTreeClassifier(random_state=SEED, max_depth=3), X_all.columns
    ),
    (
        'Random forest [all 59]',
        RandomForestClassifier(n_estimators=100, max_depth=3, random_state=SEED),
        X_all.columns
    ),
    ('XGBoost [all 59]', xgb_plain(), X_all.columns),
    ('XGBoost [VIF 47]', xgb_plain(), vif_cols_),
    ('XGBoost [corr 55]', xgb_plain(), corr_cols_),
    (
        'XGBoost, randomized search [all 59]',
        xgboost.XGBClassifier(random_state=SEED, n_jobs=1, **xgb_search.best_params_),
        X_all.columns
    ),
    ('Logistic regression [all 59]', LogisticRegression(**LOG_FIXED), X_all.columns),
    (
        'Logistic regression, grid [all 59]',
        LogisticRegression(**{**LOG_FIXED, **log_best['all 59']}), X_all.columns
    ),
    (
        'Logistic regression, grid [VIF 47]',
        LogisticRegression(**{**LOG_FIXED, **log_best['VIF 47']}), vif_cols_
    ),
    (
        'Logistic regression, grid [corr 55]',
        LogisticRegression(**{**LOG_FIXED, **log_best['corr 55']}), corr_cols_
    ),
]
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
cv_scores = {}
for lab, est, cols in CV_SPECS:
    out = cross_validate(est, X_all.loc[:, cols], y_all, cv=rskf,
                         scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=-1)
    cv_scores[lab] = {'accuracy': out['test_accuracy'], 'F1': out['test_f1'],
                      'ROC-AUC': out['test_roc_auc']}

cv_rows = []
for lab in LABELS:
    a = cv_scores[lab]['accuracy']
    obs = results.loc[lab, 'accuracy']
    cv_rows.append({'model': lab, 'committed 1234': obs, 'CV mean': a.mean(),
                    'CV sd': a.std(ddof=1), 'CV min': a.min(), 'CV max': a.max(),
                    'percentile of 1234': 100 * np.mean(a < obs)})
cv_df = pd.DataFrame(cv_rows)
print('Accuracy across %d partitions (5 folds x %d repeats), all 1,470 employees.\n'
      % (5 * N_REPEATS, N_REPEATS))
show(cv_df, '{:.4f}')

Accuracy across 50 partitions (5 folds x 10 repeats), all 1,470 employees.

                              model  committed 1234  CV mean  CV sd  CV min  CV max  percentile of 1234
             Decision tree [all 59]          0.8401   0.8449 0.0125  0.8129  0.8810             28.0000
             Random forest [all 59]          0.8401   0.8397 0.0022  0.8367  0.8435             28.0000
                   XGBoost [all 59]          0.8776   0.8690 0.0125  0.8401  0.8912             70.0000
                   XGBoost [VIF 47]          0.8810   0.8672 0.0108  0.8469  0.8980             84.0000
                  XGBoost [corr 55]          0.8741   0.8676 0.0112  0.8469  0.8946             66.0000
XGBoost, randomized search [all 59]          0.8503   0.8721 0.0147  0.8401  0.9082              4.0000
       Logistic regression [all 59]          0.8776   0.8705 0.0139  0.8401  0.8946             62.0000
 Logistic regression, grid [all 59]          0.8946   0.8799 0.0158  0.8435  0.9184         

### Why a violin, and why the committed split is drawn on it

A distribution needs a shape, and a bar with an error bar is not one. The violin shows
where the 50 fold scores actually pile up, which matters because several of these
distributions are not symmetric. The random forest's is a spike: it predicts almost no
leavers in every fold and therefore lands on the majority-class rate nearly every time.

The committed split is drawn as a single marker on each violin. That is the whole point
of the chart. The results table reports one of these dots and calls it the model's score.
Seeing the dot against the cloud it was drawn from converts an abstract caveat into an
observation the reader makes for themselves.

In [17]:
cv_order = cv_df.sort_values('CV mean')['model'].tolist()
traces = []
for lab in cv_order:
    a = cv_scores[lab]['accuracy']
    traces.append(go.Violin(
        y=[lab] * len(a), x=a, orientation='h', name=lab, side='positive', width=1.5,
        points=False, showlegend=False, line=dict(color='rgba(70,100,150,0.9)', width=1),
        fillcolor='rgba(70,100,150,0.35)',
        hovertemplate=('<b>%s</b><br>fold accuracy %%{x:.4f}<extra></extra>' % lab)
    ))
obs_x = [results.loc[l, 'accuracy'] for l in cv_order]
pctl = [cv_df.set_index('model').loc[l, 'percentile of 1234'] for l in cv_order]
traces.append(go.Scatter(
    x=obs_x, y=cv_order, mode='markers',
    marker=dict(
        size=13, color='#c0392b', symbol='diamond', line=dict(color='white', width=1.5)
    ),
    name='committed split (seed 1234)',
    text=[
        '<b>%s</b><br>committed split %.4f<br>CV mean %.4f<br>sits at the %.0fth '
        'percentile of its own 50 folds' % (l, x, cv_scores[l]['accuracy'].mean(), p)
        for l, x, p in zip(cv_order, obs_x, pctl)
    ],
    hoverinfo='text'))
traces.append(go.Scatter(
    x=[BASELINE_ACC] * 2, y=[cv_order[0], cv_order[-1]], mode='lines',
    line=dict(color='#7f8c8d', width=2, dash='dot'),
    name='predict nobody leaves (%.4f)' % BASELINE_ACC, hoverinfo='name'
))
layout = go.Layout(
    title=('Where seed 1234 falls in each model\'s own distribution of splits'
           '<br><sub>50 partitions from RepeatedStratifiedKFold(5, %d). Each fold holds '
           'out 20%%, the same proportion the committed split holds out.</sub>'
           % N_REPEATS),
    height=620, width=1000, margin=dict(l=280, r=40, t=100, b=60),
    xaxis=dict(title='accuracy', range=[0.80, 0.94]),
    yaxis=dict(tickfont=dict(size=10)),
    legend=dict(x=0.01, y=1.02, bgcolor='rgba(255,255,255,0.75)'), violingap=0.25)
py.iplot(go.Figure(data=traces, layout=layout))

In [18]:
rank_committed = results['accuracy'].rank(ascending=False, method='min').astype(int)
cvm = pd.Series({lab: cv_scores[lab]['accuracy'].mean() for lab in LABELS})
rank_cv = cvm.rank(ascending=False, method='min').astype(int)
rank_tbl = pd.DataFrame({
    'committed acc': results['accuracy'], 'rank on 1234': rank_committed,
    'CV mean acc': cvm, 'rank on 50 folds': rank_cv
})
rank_tbl['rank change'] = rank_tbl['rank on 1234'] - rank_tbl['rank on 50 folds']
print('Does the ranking survive being asked a second time?\n')
print(rank_tbl.sort_values('rank on 1234').to_string(float_format=lambda v: '%.4f' % v))
moved = int((rank_tbl['rank change'] != 0).sum())
biggest = rank_tbl['rank change'].abs().max()
print(
    '\n%d of %d models change rank when the single split is replaced by 50.'
    % (moved, len(rank_tbl))
)
print('largest single move: %d places.' % biggest)

pct = cv_df.set_index('model')['percentile of 1234']
print(
    '\nThe committed split landed between the %.0fth and the %.0fth percentile of a '
    'model\'s' % (pct.min(), pct.max())
)
print(
    'own 50 folds, depending on the model. It was not one draw, it was ten different '
    'draws.\n'
)
print('the two models the headline comparison is built on:')
for lab in (HEADLINE, RIVAL):
    print('  %-36s committed %.4f, %2.0fth percentile, CV rank %d (was %d on seed 1234)'
          % (lab, results.loc[lab, 'accuracy'], pct[lab],
             rank_tbl.loc[lab, 'rank on 50 folds'], rank_tbl.loc[lab, 'rank on 1234']))
worst_draw = pct.idxmin()
print('\nthe worst draw of the ten:')
print(
    '  %-36s committed %.4f, %2.0fth percentile, CV rank %d (was %d on seed 1234)'
    % (worst_draw, results.loc[worst_draw, 'accuracy'], pct[worst_draw],
       rank_tbl.loc[worst_draw, 'rank on 50 folds'],
       rank_tbl.loc[worst_draw, 'rank on 1234'])
)
xgbs = [l for l in LABELS if l.startswith('XGBoost')]
print('  it is also the best of the %d XGBoost variants on CV mean, and the worst of them'
      % len(xgbs))
print('  on the committed split.')

Does the ranking survive being asked a second time?

                                     committed acc  rank on 1234  CV mean acc  rank on 50 folds  rank change
Logistic regression, grid [all 59]          0.8946             1       0.8799                 1            0
Logistic regression, grid [corr 55]         0.8946             1       0.8784                 2           -1
XGBoost [VIF 47]                            0.8810             3       0.8672                 8           -5
XGBoost [all 59]                            0.8776             4       0.8690                 6           -2
Logistic regression [all 59]                0.8776             4       0.8705                 5           -1
Logistic regression, grid [VIF 47]          0.8776             4       0.8753                 3            1
XGBoost [corr 55]                           0.8741             7       0.8676                 7            0
XGBoost, randomized search [all 59]         0.8503             8       0.87

### This is the finding that changes how the table should be read

Seed 1234 was not a neutral draw, and it was not neutral in the same direction for every
model.

The **tuned XGBoost** is the clearest case. It scores 0.8503 on the committed split, which
puts it eighth of ten and reads in the results table as a hyperparameter search that made
things worse. Across 50 partitions it ranks fourth, and it is the best of the four XGBoost
variants rather than the worst. Its committed score sits at the 4th percentile of its own
distribution. It did not fail. It drew badly, and the repository's notes on the search
being reseeded read differently once that is on the page.

The **VIF XGBoost** moves the other way: third on the committed split, eighth across the
fifty. It is the biggest single move in the table, five places.

The pair the results table leads with is subtler and worth reading off the printout
above. Both the headline logistic regression and the VIF XGBoost landed at the 84th
percentile of their own fifty folds, so seed 1234 was generous to both of them by about
the same amount. What separates them is not luck on the day but what happens underneath:
across fifty partitions the headline model stays first and the VIF XGBoost falls to
eighth. The gap between them on the committed split is four employees; the gap in their
standing across fifty splits is five places. Neither number is the other's evidence.

Seven of the ten change rank when the single split is replaced by fifty. A ranking that
reorders itself when asked a second time is not a ranking.

None of this is a criticism of the seed. Any single split does this. The error is
reporting an ordering derived from one draw without the spread beside it, and that error
is in the results table of this repository.

---

## 4. The right curve for a 16% prevalence

Two curves are commonly drawn for a binary classifier, and at 16% positives they answer
noticeably different questions.

**ROC** plots recall against the false positive rate. The false positive rate has the 247
non-leavers in its denominator, so a model can raise a great many false alarms and barely
move it: eleven false positives out of 247 is 0.045. That is what makes ROC-AUC look
generous on imbalanced data. It is not wrong, it is answering "how well does this model
rank a random leaver above a random stayer", and its no-skill value is 0.5 regardless of
prevalence.

**Precision-recall** plots precision against recall. Precision has the *predicted*
positives in its denominator, so every false alarm is charged in full. Its no-skill value
is the prevalence itself, 0.1599 here. That is the number an HR partner cares about,
because it is the share of the names on the call list who are genuinely at risk.

The two disagree in tone for a mechanical reason: ROC's denominator grows with the
majority class and PR's does not. For a rare-event problem, quote PR, and quote average
precision rather than a hand-read area. Both are drawn below on the same models so the
disagreement is visible rather than asserted.

In [19]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('ROC: flattering at 16% prevalence',
                                                    'Precision-recall: the honest view'),
                    horizontal_spacing=0.09)
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
           '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for k, lab in enumerate(LABELS):
    s = y_score[lab]
    fpr, tpr, _ = roc_curve(y_true, s)
    prec, rec, _ = precision_recall_curve(y_true, s)
    auc_v = results.loc[lab, 'ROC-AUC']
    ap_v = results.loc[lab, 'avg precision']
    styl = dict(color=palette[k], width=2.6 if lab == HEADLINE else 1.5)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', line=styl, name=lab, legendgroup=lab,
        hovertemplate=('<b>%s</b><br>FPR %%{x:.3f}, recall %%{y:.3f}'
                       '<br>ROC-AUC %.4f<extra></extra>' % (lab, auc_v))
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=rec, y=prec, mode='lines', line=styl, name=lab, legendgroup=lab,
        showlegend=False,
        hovertemplate=('<b>%s</b><br>recall %%{x:.3f}, precision %%{y:.3f}'
                       '<br>average precision %.4f<extra></extra>' % (lab, ap_v))
    ), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', showlegend=False,
                         line=dict(color='black', width=1.4, dash='dot'),
                         hovertemplate='ROC no-skill = 0.5<extra></extra>'), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[PREVALENCE] * 2, mode='lines', showlegend=False,
    line=dict(color='black', width=1.4, dash='dot'),
    hovertemplate='PR no-skill = prevalence = %.4f<extra></extra>' % PREVALENCE
), row=1, col=2)
fig.update_xaxes(title='false positive rate', range=[0, 1], row=1, col=1)
fig.update_yaxes(title='recall', range=[0, 1.02], row=1, col=1)
fig.update_xaxes(title='recall', range=[0, 1], row=1, col=2)
fig.update_yaxes(title='precision', range=[0, 1.02], row=1, col=2)
fig.update_layout(
    height=560, width=1080, margin=dict(l=60, r=250, t=110, b=60),
    legend=dict(font=dict(size=9), x=1.01, y=1),
    title=('The same ten models, two curves'
           '<br><sub>Dotted lines are no skill: 0.5 for ROC, the %.4f prevalence '
           'for precision-recall. The PR baseline is a real floor; the ROC diagonal '
           'is not.</sub>' % PREVALENCE)
)
py.iplot(fig)

In [20]:
comp = pd.DataFrame({
    'ROC-AUC': results['ROC-AUC'],
    'ROC-AUC over no-skill': results['ROC-AUC'] - 0.5,
    'avg precision': results['avg precision'],
    'AP over prevalence': results['avg precision'] - PREVALENCE,
    'AP lift on prevalence': results['avg precision'] / PREVALENCE,
})
print(
    'ROC-AUC against average precision, and each one\'s distance from its own '
    'no-skill value:\n'
)
print(comp.sort_values('avg precision', ascending=False).to_string(
    float_format=lambda v: '%.4f' % v
))
print('\nThe random forest is the case worth staring at.')
rf = 'Random forest [all 59]'
print('  ROC-AUC %.4f, which sounds like a working model,'  % results.loc[rf, 'ROC-AUC'])
print(
    '  average precision %.4f against a %.4f floor,'
    % (results.loc[rf, 'avg precision'], PREVALENCE)
)
print(
    '  and it predicts %d leavers out of %d, so its precision and F1 are 0 or undefined.'
    % (int(y_hat[rf].sum()), N_TEST)
)
print('  It ranks employees usefully and classifies none of them. Only one of those two')
print('  facts survives into a table of accuracy and F1.')

ROC-AUC against average precision, and each one's distance from its own no-skill value:

                                     ROC-AUC  ROC-AUC over no-skill  avg precision  AP over prevalence  AP lift on prevalence
Logistic regression, grid [all 59]    0.8620                 0.3620         0.6643              0.5044                 4.1554
Logistic regression, grid [corr 55]   0.8565                 0.3565         0.6502              0.4903                 4.0673
XGBoost [VIF 47]                      0.8337                 0.3337         0.6451              0.4852                 4.0353
Logistic regression, grid [VIF 47]    0.8462                 0.3462         0.6225              0.4626                 3.8936
Logistic regression [all 59]          0.8570                 0.3570         0.6179              0.4580                 3.8649
Random forest [all 59]                0.8524                 0.3524         0.6074              0.4475                 3.7994
XGBoost [corr 55]            

---

## 5. Calibration: does a score of 0.7 mean a 70% chance of leaving?

The main notebook's decile table and lift chart are the most operationally useful output
in the repository, and they rest on an assumption nobody tested: that the predicted
probability means what it says. Ranking only needs the scores to be *ordered* correctly.
Saying "this employee has a 70% chance of leaving" needs them to be *calibrated*.

Two instruments:

**The reliability curve** bins the predictions and plots observed frequency against mean
predicted probability in each bin. Perfect calibration is the diagonal.

**The Brier score** is the mean squared error of the probabilities. Murphy's decomposition
splits it into three terms: *reliability* (how far the bins sit from the diagonal, lower
is better), *resolution* (how far the bin outcomes spread away from the base rate, higher
is better), and *uncertainty* (the irreducible variance of the outcome itself, which is a
property of the data and not of the model). The decomposition is exact only when the
predictions are constant within each bin, so the three terms will not sum exactly to the
Brier score, and the residual is printed rather than hidden.

The bin counts are printed alongside, and they are the reason for caution in both
directions.

In [21]:
def brier_decomposition(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    n = len(y)
    obar = y.mean()
    rel = res = 0.0
    tab = []
    for k in range(n_bins):
        m = idx == k
        nk = int(m.sum())
        if nk == 0:
            continue
        fk, ok = p[m].mean(), y[m].mean()
        rel += nk / n * (ok - fk) ** 2
        res += nk / n * (ok - obar) ** 2
        lo, hi = stats.beta.ppf([0.025, 0.975], y[m].sum() + 0.5, nk - y[m].sum() + 0.5)
        tab.append({'bin': '%.1f to %.1f' % (edges[k], edges[k + 1]), 'employees': nk,
                    'leavers': int(y[m].sum()), 'mean predicted': fk, 'observed': ok,
                    'obs lo': lo, 'obs hi': hi})
    return rel, res, obar * (1 - obar), pd.DataFrame(tab)


p_head = y_score[HEADLINE]
bs = brier_score_loss(y_true, p_head)
rel, res, unc, bin_tab = brier_decomposition(y_true, p_head)
print('%s\n' % HEADLINE)
print('Brier score              %.4f' % bs)
print('  reliability (lower better)   %.4f' % rel)
print('  resolution  (higher better)  %.4f' % res)
print('  uncertainty (fixed by data)  %.4f' % unc)
print('  rel - res + unc              %.4f   residual %+.4f (binning approximation)'
      % (rel - res + unc, bs - (rel - res + unc)))
print('\nmean predicted probability %.4f against an observed prevalence of %.4f'
      % (p_head.mean(), PREVALENCE))
print('\nReliability table, with a 95% Jeffreys interval on each bin\'s observed rate:')
show(bin_tab)

Logistic regression, grid [all 59]

Brier score              0.0837
  reliability (lower better)   0.0081
  resolution  (higher better)  0.0600
  uncertainty (fixed by data)  0.1343
  rel - res + unc              0.0824   residual +0.0013 (binning approximation)

mean predicted probability 0.1631 against an observed prevalence of 0.1599

Reliability table, with a 95% Jeffreys interval on each bin's observed rate:
       bin  employees  leavers  mean predicted  observed  obs lo  obs hi
0.0 to 0.1        190        6          0.0277    0.0316  0.0133  0.0639
0.1 to 0.2         29        4          0.1453    0.1379  0.0484  0.2954
0.2 to 0.3         19        6          0.2550    0.3158  0.1440  0.5391
0.3 to 0.4          9        0          0.3506    0.0000  0.0001  0.2376
0.4 to 0.5          9        4          0.4405    0.4444  0.1730  0.7459
0.5 to 0.6         10        6          0.5574    0.6000  0.3037  0.8469
0.6 to 0.7         11        9          0.6472    0.8182  0.5328  0.9602

In [22]:
brier_tbl = pd.DataFrame({
    'Brier': {lab: brier_score_loss(y_true, y_score[lab]) for lab in LABELS},
    'mean predicted': {lab: y_score[lab].mean() for lab in LABELS},
})
brier_tbl['prevalence'] = PREVALENCE
brier_tbl['aggregate bias'] = brier_tbl['mean predicted'] - PREVALENCE
print(
    'Brier score and aggregate bias for all ten. Lower Brier is better; '
    'aggregate bias near zero\nmeans the average predicted risk matches the observed '
    'rate.\n'
)
print(brier_tbl.sort_values('Brier').to_string(float_format=lambda v: '%.4f' % v))

Brier score and aggregate bias for all ten. Lower Brier is better; aggregate bias near zero
means the average predicted risk matches the observed rate.

                                     Brier  mean predicted  prevalence  aggregate bias
Logistic regression, grid [all 59]  0.0837          0.1631      0.1599          0.0033
Logistic regression, grid [corr 55] 0.0877          0.1645      0.1599          0.0046
Logistic regression [all 59]        0.0908          0.1614      0.1599          0.0016
XGBoost [VIF 47]                    0.0912          0.1560      0.1599         -0.0039
Logistic regression, grid [VIF 47]  0.0918          0.1655      0.1599          0.0057
XGBoost [corr 55]                   0.0962          0.1535      0.1599         -0.0063
XGBoost [all 59]                    0.0973          0.1555      0.1599         -0.0044
XGBoost, randomized search [all 59] 0.1003          0.1620      0.1599          0.0021
Random forest [all 59]              0.1124          0.1596      

### Why a reliability diagram with the histogram beneath it

A reliability curve without the score distribution under it is close to unreadable,
because it gives equal visual weight to a bin holding 190 employees and a bin holding 3.
Stacking the histogram directly below on a shared x-axis fixes that: the reader sees the
deviation and the sample size backing it in the same eye movement.

The per-bin intervals are drawn as vertical bars. They are Jeffreys intervals, which
behave sensibly when a bin contains zero or all leavers, unlike the normal approximation
which collapses to a point there. Their width is the second lesson of this section.

In [23]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32], vertical_spacing=0.06,
    subplot_titles=('Reliability: observed rate against predicted probability',
                    'Where the 294 predicted scores actually sit')
)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines', name='perfect calibration',
    line=dict(color='black', width=1.5, dash='dot'), hoverinfo='name'
), row=1, col=1)
for _, r in bin_tab.iterrows():
    fig.add_trace(go.Scatter(
        x=[r['mean predicted']] * 2, y=[r['obs lo'], r['obs hi']], mode='lines',
        line=dict(color='rgba(192,57,43,0.55)', width=2.5), showlegend=False,
        hoverinfo='skip'
    ), row=1, col=1)
fig.add_trace(go.Scatter(
    x=bin_tab['mean predicted'], y=bin_tab['observed'], mode='markers+lines',
    marker=dict(size=np.clip(6 + bin_tab['employees'] * 0.35, 7, 26), color='#c0392b',
                line=dict(color='white', width=1.2)),
    line=dict(color='rgba(192,57,43,0.6)', width=1.5), name=HEADLINE,
    text=['bin %s<br>%d employees, %d leavers<br>mean predicted %.4f<br>observed %.4f'
          '<br>95%% interval [%.4f, %.4f]'
          % (r['bin'], r['employees'], r['leavers'], r['mean predicted'], r['observed'],
             r['obs lo'], r['obs hi']) for _, r in bin_tab.iterrows()],
    hoverinfo='text'), row=1, col=1)
fig.add_trace(go.Histogram(x=p_head, xbins=dict(start=0, end=1, size=0.05),
                           marker=dict(color='#16406e'), name='predicted score',
                           hovertemplate='score %{x}<br>%{y} employees<extra></extra>',
                           showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(
    x=[PREVALENCE] * 2, y=[0, 1], mode='lines', showlegend=False,
    line=dict(color='#7f8c8d', width=1.5, dash='dash'),
    hovertemplate='prevalence %.4f<extra></extra>' % PREVALENCE
), row=1, col=1)
fig.update_xaxes(range=[0, 1], row=1, col=1)
fig.update_yaxes(title='observed leaver rate', range=[-0.03, 1.03], row=1, col=1)
fig.update_xaxes(title='predicted probability of leaving', range=[0, 1], row=2, col=1)
fig.update_yaxes(title='employees', row=2, col=1)
fig.update_layout(
    height=720, width=880, margin=dict(l=70, r=40, t=110, b=60), bargap=0.02,
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.75)'),
    title=('Are these probabilities probabilities?'
           '<br><sub>Marker size is bin count. Bars are 95%% Jeffreys intervals. '
           'Brier %.4f, reliability term %.4f.</sub>' % (bs, rel))
)
py.iplot(fig)

In [24]:
top_bins = bin_tab[bin_tab['mean predicted'] > 0.5]
print('The scores are well calibrated in aggregate and unverifiable at the top.\n')
print('In aggregate: mean predicted %.4f against observed %.4f, a bias of %+.4f,'
      % (p_head.mean(), PREVALENCE, p_head.mean() - PREVALENCE))
print(
    'and a reliability term of %.4f out of a Brier score of %.4f, which is %.1f%% of it.'
    % (rel, bs, 100 * rel / bs)
)
print('\nAt the top of the range, where the decile targeting story lives:')
print(
    '  %d employees in total sit above a predicted 0.5' % int(top_bins['employees'].sum())
)
print('  spread over %d bins holding %s employees each'
      % (len(top_bins), ', '.join(str(int(v)) for v in top_bins['employees'])))
print('  the widest of those bin intervals spans %.4f'
      % (top_bins['obs hi'] - top_bins['obs lo']).max())
worst = bin_tab.loc[(bin_tab['observed'] - bin_tab['mean predicted']).abs().idxmax()]
print('\nThe worst-looking bin is %s: predicted %.4f, observed %.4f.'
      % (worst['bin'], worst['mean predicted'], worst['observed']))
print('It holds %d employees and its 95%% interval runs [%.4f, %.4f], which contains the'
      % (worst['employees'], worst['obs lo'], worst['obs hi']))
print('predicted value. The wobble in the curve is what %d-employee bins look like, not'
      % worst['employees'])
print('evidence of miscalibration.')

top30 = np.argsort(-p_head)[:30]
k30 = int(y_true[top30].sum())
lo30, hi30 = stats.beta.ppf([0.025, 0.975], k30 + 0.5, 30 - k30 + 0.5)
print(
    '\nThe first decile, which is the number the main notebook asks an HR partner to '
    'act on:'
)
print('  the top %d employees by predicted score contain %d leavers, a hit rate of %.4f'
      % (30, k30, k30 / 30))
print('  95%% Jeffreys interval on that hit rate: [%.4f, %.4f]' % (lo30, hi30))
print('  it is a real and useful number, and it is not accurate to four decimals.')

The scores are well calibrated in aggregate and unverifiable at the top.

In aggregate: mean predicted 0.1631 against observed 0.1599, a bias of +0.0033,
and a reliability term of 0.0081 out of a Brier score of 0.0837, which is 9.7% of it.

At the top of the range, where the decile targeting story lives:
  38 employees in total sit above a predicted 0.5
  spread over 5 bins holding 10, 11, 9, 3, 5 employees each
  the widest of those bin intervals spans 0.5729

The worst-looking bin is 0.3 to 0.4: predicted 0.3506, observed 0.0000.
It holds 9 employees and its 95% interval runs [0.0001, 0.2376], which contains the
predicted value. The wobble in the curve is what 9-employee bins look like, not
evidence of miscalibration.

The first decile, which is the number the main notebook asks an HR partner to act on:
  the top 30 employees by predicted score contain 23 leavers, a hit rate of 0.7667
  95% Jeffreys interval on that hit rate: [0.5956, 0.8891]
  it is a real and useful number, and it 

### What this means for the decile story

The verdict is a qualified yes, and the qualification matters.

**In aggregate the probabilities are honest.** The mean predicted risk lands within a
third of a percentage point of the observed rate, and the reliability term is under a
tenth of the Brier score. Most of the Brier score is irreducible uncertainty, which no
model can remove.

**Per band, the claim is untestable here.** Above a predicted 0.5 there are 38 employees
in total, spread across five bins holding between three and eleven people each. Every
interval on those bins is wide enough to contain the diagonal and a good deal else
besides. The reliability curve wobbles above and below the line at the top of the range,
and at those counts that wobble is what random variation looks like, not a defect.

So the sentence to attach to the decile table is not "the scores are miscalibrated". It
is: **the aggregate calibration checks out, and the per-band rates the table quotes are
each estimated from single-digit or low-double-digit counts.** The first decile's hit rate
is 23 leavers among the top 30 employees, and the interval printed above runs from roughly
0.60 to 0.89. That is a genuinely useful operational number. Plan the week around it, and
do not quote the fourth decimal of it.

---

## 6. Selection optimism: what is best-of-800 worth?

The randomized search fitted 800 candidate XGBoost configurations, scored each by 5-fold
cross-validated F1 on the training set, and took the maximum. The maximum of 800 noisy
estimates is not an unbiased estimate of the winner's quality. It is biased upward by
construction, because a configuration that got a lucky set of folds is more likely to be
the one selected. This is the same mechanism as the winner's curse in auctions, and it is
why a cross-validated score reported for a searched model is a *selection* criterion and
not a performance estimate.

Three quantities make the size of the problem concrete: the gap between the winner and the
runner-up, the fold-to-fold standard error of the winner's own score, and the gap between
the winner's cross-validated score and what it actually achieved on the held-out set.

In [25]:
cvres = pd.DataFrame(xgb_search.cv_results_).sort_values('rank_test_score')
fold_cols = [
    c for c in cvres.columns if c.startswith('split') and c.endswith('test_score')
]
winner, runner = cvres.iloc[0], cvres.iloc[1]
fold_sd = float(winner[fold_cols].astype(float).std(ddof=1))
fold_se = fold_sd / np.sqrt(len(fold_cols))
test_f1 = results.loc['XGBoost, randomized search [all 59]', 'F1']
gap = float(winner['mean_test_score'] - runner['mean_test_score'])
within = int((cvres['mean_test_score'] >= winner['mean_test_score'] - fold_se).sum())

print('Randomized search over %d candidates, 5-fold CV on F1.\n' % N_CANDIDATES)
print('winner    mean CV F1  %.6f' % winner['mean_test_score'])
print('runner-up mean CV F1  %.6f' % runner['mean_test_score'])
print('gap                   %.6f' % gap)
print(
    '\nwinner fold-to-fold sd %.4f, standard error over 5 folds %.4f' % (fold_sd, fold_se)
)
print('the gap is %.3f standard errors, or 1 part in %.0f of one standard error'
      % (gap / fold_se, fold_se / gap))
print(
    '\ncandidates within one standard error of the winner: %d of %d'
    % (within, N_CANDIDATES)
)
print('CV F1 range across all %d candidates: %.4f to %.4f'
      % (N_CANDIDATES, cvres['mean_test_score'].min(), cvres['mean_test_score'].max()))
print('\nwinner CV F1        %.4f' % winner['mean_test_score'])
print('winner test-set F1  %.4f' % test_f1)
print('optimism            %+.4f' % (winner['mean_test_score'] - test_f1))
print(
    '\nfor contrast, the untuned XGBoost on the same features scored %.4f on the same '
    'test set' % results.loc['XGBoost [all 59]', 'F1']
)

Randomized search over 800 candidates, 5-fold CV on F1.

winner    mean CV F1  0.491827
runner-up mean CV F1  0.491416
gap                   0.000411

winner fold-to-fold sd 0.0700, standard error over 5 folds 0.0313
the gap is 0.013 standard errors, or 1 part in 76 of one standard error

candidates within one standard error of the winner: 19 of 800
CV F1 range across all 800 candidates: 0.0000 to 0.4918

winner CV F1        0.4918
winner test-set F1  0.4500
optimism            +0.0418

for contrast, the untuned XGBoost on the same features scored 0.4706 on the same test set


In [26]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.10,
                    subplot_titles=('All %d candidates by mean CV F1' % N_CANDIDATES,
                                    'The winner: CV score against test score'))
fig.add_trace(go.Histogram(
    x=cvres['mean_test_score'], nbinsx=60, marker=dict(color='#16406e'),
    name='candidates', showlegend=False,
    hovertemplate='CV F1 %{x}<br>%{y} candidates<extra></extra>'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[winner['mean_test_score']] * 2, y=[0, 60], mode='lines',
    line=dict(color='#c0392b', width=2), showlegend=False,
    hovertemplate='winner %.6f<extra></extra>' % winner['mean_test_score']
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[winner['mean_test_score'] - fold_se, winner['mean_test_score'] + fold_se],
    y=[30, 30],
    mode='lines', line=dict(color='rgba(192,57,43,0.45)', width=9), showlegend=False,
    hovertemplate=(
        'one standard error of the winner: %.4f<br>%d of %d candidates fall inside'
        '<extra></extra>' % (fold_se, within, N_CANDIDATES)
    )
), row=1, col=1)
fold_vals = winner[fold_cols].astype(float).values
fig.add_trace(go.Scatter(
    x=['CV folds'] * len(fold_vals), y=fold_vals, mode='markers',
    marker=dict(size=11, color='rgba(22,64,110,0.65)'), showlegend=False,
    hovertemplate='fold F1 %{y:.4f}<extra></extra>'
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=['CV folds'], y=[winner['mean_test_score']], mode='markers',
    marker=dict(size=16, color='#16406e', symbol='diamond'), showlegend=False,
    hovertemplate='mean CV F1 %{y:.4f}<extra></extra>'
), row=1, col=2)
fig.add_trace(go.Scatter(x=['held-out 294'], y=[test_f1], mode='markers',
                         marker=dict(size=16, color='#c0392b', symbol='diamond'),
                         showlegend=False,
                         hovertemplate='test F1 %{y:.4f}<extra></extra>'), row=1, col=2)
lo_b, hi_b = pct_ci(boot['XGBoost, randomized search [all 59]']['F1'])
fig.add_trace(go.Scatter(
    x=['held-out 294'] * 2, y=[lo_b, hi_b], mode='lines',
    line=dict(color='rgba(192,57,43,0.5)', width=9), showlegend=False,
    hovertemplate=('bootstrap 95%% CI [%.4f, %.4f]<extra></extra>' % (lo_b, hi_b))
), row=1, col=2)
fig.update_xaxes(title='mean cross-validated F1', row=1, col=1)
fig.update_yaxes(title='candidates', row=1, col=1)
fig.update_yaxes(title='F1', range=[0.25, 0.75], row=1, col=2)
fig.update_layout(
    height=520, width=1020, margin=dict(l=70, r=40, t=110, b=60),
    title=('Best of %d is a selection, not an estimate'
           '<br><sub>The winner beat the runner-up by %.6f. One standard error of '
           'the winner\'s own score is %.4f, and %d candidates sit inside it.</sub>'
           % (N_CANDIDATES, gap, fold_se, within))
)
py.iplot(fig)

### What a reader should do about it

The winner beat the runner-up by 0.0004 of mean cross-validated F1, on a score whose own
fold-to-fold standard error is more than seventy times that gap. Nineteen of the 800
candidates sit within one standard error of the top. Re-running the search with a
different fold seed would return a different winner from that group, and the section on
split sensitivity has already shown what a different draw does.

Two responses, and they are not alternatives:

**Treat the cross-validated score as selection, never as estimation.** It picked a
configuration. It does not estimate that configuration's performance, because the same
data both chose and scored it. The gap here is +0.0418 of F1, and the held-out set is the
only clean read available. This is cheap and should be the default.

**Use nested cross-validation when the estimate itself matters.** Put the entire search
inside an outer loop so that each outer fold's estimate comes from data the search never
touched. It costs an outer-fold multiple of the search time, which for this search is
minutes rather than hours, and it is the only way to report a searched model's performance
without the winner's curse baked in.

And a third response that costs nothing: **report the runner-up.** A search that returns
"0.4918, and 19 candidates within one standard error" tells the reader what a search that
returns "0.4918" hides.

---

## 7. What the coefficients actually say

The main notebook plots logistic regression coefficients sorted by magnitude and reads
the largest as the most important features. That reading is only valid when the features
share a scale, and these do not. The design matrix mixes 0/1 dummies with `MonthlyIncome`
in currency units and `MonthlyRate` in another. A coefficient is the log-odds change per
**one unit** of its feature, and one unit means something completely different in each
column.

Below: the standard deviation of every feature in the training set, the raw coefficient,
and the coefficient multiplied by that standard deviation, which is the log-odds change
per one-standard-deviation move and is at least comparable across columns.

Then the honest alternative. **Permutation importance** shuffles one column of the test
set, re-scores the fitted model, and measures how much the score drops. It needs no
assumption about scale, it measures the feature's contribution to *this model's
predictions on held-out data*, and it comes with its own spread because the shuffling is
random. It is computed here against ROC-AUC, which uses the full ranking rather than the
0.5 cut-off.

In [27]:
head_model = fitted[HEADLINE]
coefs = head_model.coef_.ravel()
feat_sd = X_train.std().values
N_PERM = 50
perm = permutation_importance(head_model, X_test, y_test, scoring='roc_auc',
                              n_repeats=N_PERM, random_state=SEED, n_jobs=-1)
imp = pd.DataFrame({
    'feature': X_train.columns, 'coefficient': coefs, 'feature sd': feat_sd,
    'coef x sd': coefs * feat_sd, 'perm importance': perm.importances_mean,
    'perm sd': perm.importances_std
})
imp['perm lo'] = imp['perm importance'] - 1.96 * imp['perm sd'] / np.sqrt(N_PERM)
imp['perm hi'] = imp['perm importance'] + 1.96 * imp['perm sd'] / np.sqrt(N_PERM)

print(
    'Feature standard deviations in the training set range from %.4f to %.1f, a factor '
    'of %.0f.' % (feat_sd.min(), feat_sd.max(), feat_sd.max() / feat_sd.min())
)
print(
    'A coefficient is per one unit of its own feature, so these are not comparable '
    'numbers.\n'
)
print('Top 8 by raw |coefficient|, the ordering the main notebook plots:')
show(imp.reindex(imp['coefficient'].abs().sort_values(ascending=False).index)
        .head(8)[[
            'feature', 'coefficient', 'feature sd', 'coef x sd', 'perm importance'
        ]])
print('\nTop 8 by permutation importance on the held-out set:')
show(imp.reindex(imp['perm importance'].sort_values(ascending=False).index)
        .head(8)[['feature', 'perm importance', 'perm sd', 'coefficient', 'coef x sd']])

Feature standard deviations in the training set range from 0.1791 to 7101.8, a factor of 39644.
A coefficient is per one unit of its own feature, so these are not comparable numbers.

Top 8 by raw |coefficient|, the ordering the main notebook plots:
                         feature  coefficient  feature sd  coef x sd  perm importance
       JobRole_Research Director      -3.0798      0.2236    -0.6885           0.0213
                     compa_ratio      -2.7889      0.2289    -0.6383           0.0221
         JobRole_Human Resources       2.7293      0.1899     0.5182           0.0057
                    OverTime_Yes       2.1604      0.4495     0.9711           0.0812
    JobRole_Sales Representative       2.1189      0.2236     0.4737           0.0434
   JobRole_Laboratory Technician       1.8307      0.3817     0.6988           0.0259
BusinessTravel_Travel_Frequently       1.7027      0.3803     0.6475           0.0428
                        JobLevel      -1.6519      1.0962    -

In [28]:
mi = imp[imp.feature == 'MonthlyIncome'].iloc[0]
rank_coef = int(imp['coefficient'].abs().rank(ascending=False)[
    imp.feature == 'MonthlyIncome'
].iloc[0])
rank_std = int(imp['coef x sd'].abs().rank(ascending=False)[
    imp.feature == 'MonthlyIncome'
].iloc[0])
rank_perm = int(imp['perm importance'].rank(ascending=False)[
    imp.feature == 'MonthlyIncome'
].iloc[0])
print('MonthlyIncome is the cleanest demonstration of the problem.\n')
print('  raw coefficient          %.6f   -> rank %d of %d by magnitude'
      % (mi['coefficient'], rank_coef, len(imp)))
print('  feature sd               %.1f' % mi['feature sd'])
print(
    '  coefficient x sd         %.4f   -> rank %d of %d'
    % (mi['coef x sd'], rank_std, len(imp))
)
print(
    '  permutation importance   %.4f   -> rank %d of %d'
    % (mi['perm importance'], rank_perm, len(imp))
)
print(
    '\nRead off the raw coefficient it looks like one of the least important features '
    'in the'
)
print(
    'model. Rescaled, it has the largest standardised effect of all %d. The coefficient '
    'was' % len(imp)
)
print('never small; the unit was.')
nz = int((imp['perm lo'] > 0).sum())
print(
    '\n%d of %d features have a permutation importance whose interval clears zero, '
    'uncorrected' % (nz, len(imp))
)
print(
    'for %d simultaneous comparisons. The bars below measure shuffling noise only, not '
    'the' % len(imp)
)
print('sampling noise of the 294 employees, so they are narrower than the truth.')

MonthlyIncome is the cleanest demonstration of the problem.

  raw coefficient          0.000477   -> rank 57 of 59 by magnitude
  feature sd               4614.9
  coefficient x sd         2.2021   -> rank 1 of 59
  permutation importance   0.0711   -> rank 3 of 59

Read off the raw coefficient it looks like one of the least important features in the
model. Rescaled, it has the largest standardised effect of all 59. The coefficient was
never small; the unit was.

34 of 59 features have a permutation importance whose interval clears zero, uncorrected
for 59 simultaneous comparisons. The bars below measure shuffling noise only, not the
sampling noise of the 294 employees, so they are narrower than the truth.


### Why these two are drawn side by side rather than described

The claim is that two orderings of the same 59 features disagree. A reader can be told
that, or shown it, and shown is shorter. Two horizontal bar panels sharing a y-axis, with
the features ordered by permutation importance, put each feature's two answers on the same
line. Rows where the bars disagree in length are the argument, and `MonthlyIncome` is the
one to look for: a substantial bar on the left, a bar of visually zero length on the
right.

Error bars appear only on the left panel, because only permutation importance has a
spread to draw. That asymmetry is itself informative: the coefficient plot in the main
notebook has no uncertainty attached to it at all, and none is available without
refitting.

In [29]:
TOP_N = 18
top = imp.reindex(imp['perm importance'].sort_values(ascending=False).index).head(TOP_N)
top = top.iloc[::-1]
fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
    subplot_titles=('Permutation importance on the held-out 294 (drop in ROC-AUC)',
                    'Raw logistic coefficient, the units it was fitted in')
)
fig.add_trace(go.Bar(
    y=top['feature'], x=top['perm importance'], orientation='h',
    marker=dict(color='#16406e'), showlegend=False,
    error_x=dict(type='data', symmetric=False,
                 array=(top['perm hi'] - top['perm importance']).values,
                 arrayminus=(top['perm importance'] - top['perm lo']).values,
                 color='rgba(0,0,0,0.55)', thickness=1.4, width=3),
    customdata=np.stack([top['perm sd'], top['perm lo'], top['perm hi']], axis=-1),
    hovertemplate=('<b>%{y}</b><br>ROC-AUC drops %{x:.4f} when shuffled'
                   '<br>sd across %d shuffles %%{customdata[0]:.4f}'
                   '<br>interval [%%{customdata[1]:.4f}, %%{customdata[2]:.4f}]'
                   '<extra></extra>').replace('%d', str(N_PERM), 1)
), row=1, col=1)
fig.add_trace(go.Bar(
    y=top['feature'], x=top['coefficient'], orientation='h',
    marker=dict(color=np.where(top['coefficient'] > 0, '#c0392b', '#7f8c8d')),
    showlegend=False,
    customdata=np.stack([top['feature sd'], top['coef x sd']], axis=-1),
    hovertemplate=('<b>%{y}</b><br>coefficient %{x:.4f} per unit'
                   '<br>feature sd %{customdata[0]:.3f}'
                   '<br>coefficient x sd %{customdata[1]:.4f}<extra></extra>')
), row=1, col=2)
fig.update_xaxes(title='drop in ROC-AUC when shuffled', row=1, col=1)
fig.update_xaxes(title='log-odds per one unit of the feature', row=1, col=2)
fig.update_yaxes(tickfont=dict(size=10), row=1, col=1)
fig.update_layout(
    height=680, width=1060, margin=dict(l=250, r=40, t=110, b=60),
    title=('Two answers to "which features matter", from the same fitted model'
           '<br><sub>Ordered by permutation importance. MonthlyIncome is the row '
           'to look at: it moves ROC-AUC by %.4f and has a coefficient of %.6f.</sub>'
           % (mi['perm importance'], mi['coefficient']))
)
py.iplot(fig)

---

## 8. Would more data help?

This is the question a manager asks once they have been told the answer is inconclusive,
and it deserves a real answer rather than a shrug.

A learning curve refits the model on growing subsets of the data and plots training and
validation score against training-set size. The shape carries the diagnosis. A validation
curve that has flattened means more rows of the same kind will not help and the limit is
the model or the features. A validation curve still climbing at the right-hand edge means
the model is data-limited and more rows would buy something.

The band matters as much as the line. Each point below is the mean over 50 partitions,
and the shaded region is the 10th to 90th percentile of those partitions, which shows how
much the answer moves depending on which employees happen to land in the fold. A learning
curve drawn as a single line hides exactly the variability this notebook is about.

Scored on ROC-AUC rather than accuracy, because accuracy at 16% prevalence is dominated by
the majority class and would look flat regardless.

In [30]:
train_sizes = np.linspace(0.15, 1.0, 8)
sizes, train_sc, val_sc = learning_curve(
    LogisticRegression(**{**LOG_FIXED, **log_best['all 59']}), X_all, y_all,
    train_sizes=train_sizes, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS,
                                                        random_state=SEED),
    scoring='roc_auc', n_jobs=-1, shuffle=True, random_state=SEED)

lc = pd.DataFrame({
    'training rows': sizes.astype(int),
    'train ROC-AUC': train_sc.mean(1),
    'validation ROC-AUC': val_sc.mean(1),
    'validation sd': val_sc.std(1, ddof=1),
    'val p10': np.percentile(val_sc, 10, axis=1),
    'val p90': np.percentile(val_sc, 90, axis=1),
})
lc['gain over previous'] = lc['validation ROC-AUC'].diff()
lc['band width'] = lc['val p90'] - lc['val p10']
show(lc)
last_gain = lc['gain over previous'].iloc[-1]
last_band = lc['band width'].iloc[-1]
step_rows = int(sizes[-1] - sizes[-2])
print(
    '\nThe last %d training rows bought %+.4f of validation ROC-AUC.'
    % (step_rows, last_gain)
)
print('The fold-to-fold band at that point is %.4f wide, which is %.0f times the gain.'
      % (last_band, last_band / last_gain))
if last_gain > 0:
    rows_needed = 0.02 / (last_gain / step_rows)
    print(
        'At the current slope, moving ROC-AUC by 0.02 would take roughly %.0f more '
        'employees,' % rows_needed
    )
    print('and the band would still be wider than the improvement.')

 training rows  train ROC-AUC  validation ROC-AUC  validation sd  val p10  val p90  gain over previous  band width
           176         0.9987              0.7168         0.0511   0.6331   0.7829                 NaN      0.1498
           319         0.9593              0.7739         0.0385   0.7268   0.8206              0.0571      0.0938
           462         0.9332              0.8048         0.0386   0.7615   0.8551              0.0309      0.0936
           604         0.9161              0.8169         0.0370   0.7816   0.8707              0.0122      0.0892
           747         0.9049              0.8261         0.0371   0.7878   0.8746              0.0092      0.0868
           890         0.8967              0.8316         0.0374   0.7928   0.8824              0.0055      0.0896
          1033         0.8915              0.8375         0.0367   0.7988   0.8879              0.0058      0.0891
          1176         0.8876              0.8405         0.0368   0.8011   0.88

In [31]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.concatenate([lc['training rows'], lc['training rows'][::-1]]),
    y=np.concatenate([lc['val p90'], lc['val p10'][::-1]]),
    fill='toself', fillcolor='rgba(22,64,110,0.16)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip',
    name='validation, 10th to 90th percentile of 50 folds'
))
fig.add_trace(go.Scatter(
    x=lc['training rows'], y=lc['validation ROC-AUC'], mode='markers+lines',
    line=dict(color='#16406e', width=3), marker=dict(size=9),
    name='validation (held-out folds)',
    customdata=np.stack([lc['val p10'], lc['val p90'], lc['validation sd']], axis=-1),
    hovertemplate=('%{x} training rows<br>validation ROC-AUC %{y:.4f}'
                   '<br>10th to 90th percentile [%{customdata[0]:.4f}, '
                   '%{customdata[1]:.4f}]<br>sd across folds '
                   '%{customdata[2]:.4f}<extra></extra>')
))
fig.add_trace(go.Scatter(
    x=lc['training rows'], y=lc['train ROC-AUC'], mode='markers+lines',
    line=dict(color='#c0392b', width=2, dash='dash'), marker=dict(size=7),
    name='training (in-sample)',
    hovertemplate='%{x} training rows<br>training ROC-AUC %{y:.4f}<extra></extra>'
))
fig.add_trace(go.Scatter(x=[1176, 1176], y=[0.65, 1.0], mode='lines',
                         line=dict(color='#7f8c8d', width=1.5, dash='dot'),
                         name='the committed training set (1,176)', hoverinfo='name'))
fig.update_layout(
    height=560, width=960, margin=dict(l=70, r=40, t=110, b=60),
    xaxis=dict(title='training rows'), yaxis=dict(title='ROC-AUC', range=[0.65, 1.01]),
    legend=dict(x=0.42, y=0.30, bgcolor='rgba(255,255,255,0.8)'),
    title=('Would more employees help?'
           '<br><sub>Grid-searched logistic regression, 50 partitions per point. '
           'The validation curve is still rising at 1,176 rows, and the last %d rows '
           'bought %+.4f.</sub>'
           % (step_rows, last_gain)))
py.iplot(fig)

### The answer, in the form a manager can use

**Yes, and not enough to matter for the question you asked.**

The validation curve has not flattened at 1,176 training rows, so this model is still
data-limited: more employees of the same kind would make it genuinely better. But the
slope over the last step is small, the fold-to-fold band around it is many times wider
than the gain, and the training curve has fallen close enough to the validation curve
that there is no large overfitting gap left to close either.

More importantly, the learning curve is answering the wrong question for this notebook.
Adding training rows improves the model. It does not shrink the error bar on the
*evaluation*, which is set by the number of held-out **events**, and there are 47.
Section 2 costed that directly: separating the top two models at conventional power needs
a held-out set several times the size of this entire dataset.

If a decision hangs on which of these models to deploy, the answer is not "collect more
data and re-run this notebook". Either the choice does not matter, and should be made on
grounds the data cannot speak to, which for these two models means auditability, latency
and the cost of maintaining an XGBoost search. Or the evaluation needs to be of a
different kind altogether: a prospective trial, or a survival model with a time
dimension, on real employees rather than a demo file.

---

## What the eight tests add up to

Restating the ground rule one last time, because it governs everything above: **these
1,470 employees are fictional**, generated by IBM for a Watson Analytics demo. Every
number in this notebook is a measurement of a method applied to a synthetic file. None of
it is a finding about attrition, at any company, including the parts that sound like
common sense.

Within that boundary, the results are unambiguous.

**What holds.** Not the baseline claim, as section 1c shows: eight of the ten post a
higher accuracy than 0.8401, and none of the ten survives correction against it. What
holds is ranking rather than classification. The grid-searched logistic regression flags
38 employees, 27 of whom left, against a base rate of 16%. Its scores are calibrated in
aggregate. The top decile really does concentrate risk, and the cumulative gain chart in
the main notebook is the most useful thing in the repository. Sorting employees by
predicted risk works.

**What does not hold.** The ordering of the ten rows in the results table. Not one of the
45 pairwise comparisons survives multiplicity correction, and the fourteen that reach even
nominal significance all involve one of the three weakest rows. Every model's 95% accuracy
interval is wider than the entire spread of the ten point estimates. Seven of the ten
change rank when the single split is replaced by fifty. The tuned XGBoost, which the table
places eighth and which the repository's notes treat as a search that went wrong, ranks
fourth across fifty partitions and is the strongest of the four XGBoost variants there.

**The distinction to keep.** A ranking sorts. A result survives being asked twice. This
table sorts, and the notebook that produced it presented the sort as though it were the
second thing. That is a mistake made by nearly every model comparison table published,
and it is made here in a repository other people have copied, which is the reason to fix
it in public rather than quietly.

**The concrete recommendation.** Report the group, not the row. Strictly, after
correction, nothing here separates any pair of the ten. The strongest defensible claim is
about ordering rather than labelling: the scores concentrate risk, with 23 of the top 30
being leavers. The ten cannot be ranked among themselves on 47 events. Choose within that
group on grounds the evaluation is not being asked to rank. The grid-searched logistic
regression is 59 coefficients an HR director can read, it needs no search to reproduce,
and it is calibrated in aggregate. Those are better reasons to deploy it than four
employees.

---

## The summary card

One image, matplotlib rather than plotly because this one is meant to be exported and
pasted somewhere without a JavaScript runtime. Every number on it is read out of the
variables computed above, so it cannot drift away from the notebook.

Laid out with `gridspec` and `constrained_layout` rather than hand-placed coordinates, so
that a different font or a different backend reflows it instead of overlapping it.

In [32]:
OUT_PNG = '../img/statistical_rigour_summary.png'
# This notebook owns this filename and rewrites it on every run, which is exactly what
# the reproduce command at the end of this notebook does. The guard is against OUT_PNG
# being pointed at a figure another notebook owns, not against regenerating our own.
OWNED_ELSEWHERE = {
    'Cum_gain.png',
    'HRM.jpg',
    'Lift.png',
    'Question3_viz.png',
    'Sample_univariate.JPG',
    'confusion-matrix.png',
    'emergency-exit.jpg',
    'employee-attrition.jpg',
    'hr_correlation_matrix.png',
    'random-forest.png',
    'xgboost.png',
}
assert os.path.basename(OUT_PNG) not in OWNED_ELSEWHERE, \
    'refusing to write over a figure another notebook owns: %s' % OUT_PNG

plt.style.use('default')
INK, BLUE, RED, GREY = '#1c2833', '#16406e', '#c0392b', '#95a5a6'
fig = plt.figure(figsize=(15.5, 11.5), dpi=140, constrained_layout=True)
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(3, 2, figure=fig, height_ratios=[0.40, 1.0, 0.82],
                       width_ratios=[1.25, 1.0])

# --- header
ax = fig.add_subplot(gs[0, :]); ax.axis('off')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.text(0, 0.96, 'Can %d employees tell ten models apart?' % N_TEST,
        fontsize=27, fontweight='bold', color=INK, va='top')
ax.text(
    0, 0.50,
    'Almost certainly not. %d of the %d held-out employees left. The headline gap '
    'in the results table is %d of them.'
    % (
        N_POS, N_TEST, int(results.loc[RIVAL, 'errors'] - results.loc[HEADLINE, 'errors'])
    ),
    fontsize=14.5, color=INK, va='top'
)
ax.text(
    0, 0.20,
    'Fictional IBM sample data. This measures the method and says nothing about '
    'any real workforce.', fontsize=12, color=RED, style='italic', va='top'
)

# --- panel A: forest of accuracy
axA = fig.add_subplot(gs[1, 0])
ypos = np.arange(len(order))
axA.hlines(ypos, lo, hi, color=BLUE, alpha=0.75, linewidth=3.2)
axA.plot(
    pt, ypos, 'o', color=BLUE, markersize=8, markeredgecolor='white', markeredgewidth=1.1
)
axA.axvline(BASELINE_ACC, color=RED, linestyle='--', linewidth=1.8)
axA.text(
    BASELINE_ACC - 0.004, len(order) - 1.45, 'predict nobody leaves\n%.4f' % BASELINE_ACC,
    color=RED, fontsize=10, ha='right', va='top'
)
axA.set_ylim(-0.8, len(order) - 0.2)
axA.set_yticks(ypos); axA.set_yticklabels(order, fontsize=10.5)
axA.set_xlabel(
    'accuracy on the held-out %d, with 95%% bootstrap interval' % N_TEST, fontsize=11.5
)
axA.set_title('Ten models, ten overlapping intervals\n%d bootstrap draws, seed %d'
              % (B_BOOT, SEED), fontsize=14, fontweight='bold', color=INK, loc='left')
axA.set_xlim(min(lo) - 0.025, max(hi) + 0.02)
axA.grid(axis='x', alpha=0.25); axA.set_axisbelow(True)
for s in ('top', 'right', 'left'):
    axA.spines[s].set_visible(False)

# --- panel B: the 294 employees, and the 26 the top two models disagree about
axB = fig.add_subplot(gs[1, 1])
cols_grid = 21
rows_grid = int(np.ceil(N_TEST / cols_grid))
gx = np.array([i % cols_grid for i in range(N_TEST)], dtype=float)
gy = np.array([rows_grid - 1 - i // cols_grid for i in range(N_TEST)], dtype=float)
stayed = y_true == 0
axB.scatter(
    gx[stayed], gy[stayed], s=66, color='#d5dbdb', edgecolor='white', linewidth=0.7
)
axB.scatter(gx[~stayed], gy[~stayed], s=66, color=BLUE, edgecolor='white', linewidth=0.7)
head_right = correct[HEADLINE]
rival_right = correct[RIVAL]
win_ix = np.where(head_right & ~rival_right)[0]
lose_ix = np.where(~head_right & rival_right)[0]
axB.scatter(
    gx[win_ix], gy[win_ix], s=215, facecolor='none', edgecolor='#1e8449', linewidth=2.4
)
axB.scatter(
    gx[lose_ix], gy[lose_ix], s=215, facecolor='none', edgecolor=RED, linewidth=2.4
)
n_gap = len(win_ix) - len(lose_ix)
axB.set_xlim(-1.2, cols_grid + 0.2); axB.set_ylim(-5.4, rows_grid + 0.2)
axB.axis('off')
axB.set_title('The entire evidence base\n%d employees, %d of whom left'
              % (N_TEST, N_POS), fontsize=14, fontweight='bold', color=INK, loc='left')
axB.text(-1.0, -1.4, 'grey: stayed (%d)          blue fill: left (%d)'
         % (int(stayed.sum()), N_POS), fontsize=10.5, color=INK, va='top')
axB.text(
    -1.0, -2.9,
    'The top two rows of the table disagree about %d of these employees:\n'
    '%d go to the headline model (green ring), %d to the VIF XGBoost (red\n'
    'ring). Net %d. Exact McNemar p = %.4f.'
    % (len(win_ix) + len(lose_ix), len(win_ix), len(lose_ix), n_gap, row['p']),
    fontsize=10.5, color=INK, va='top', linespacing=1.5
)

# --- panel C: McNemar tally
axC = fig.add_subplot(gs[2, 0])
bars = ['comparisons run', 'raw p < 0.05', 'survive Holm']
vals = [len(mcnemar_df), n_raw, n_holm]
colors = [GREY, '#e67e22', RED if n_holm else BLUE]
b_ = axC.barh(bars[::-1], vals[::-1], color=colors[::-1], height=0.42)
for rect, v in zip(b_, vals[::-1]):
    axC.text(rect.get_width() + 0.7, rect.get_y() + rect.get_height() / 2, str(v),
             va='center', fontsize=16, fontweight='bold', color=INK)
axC.set_xlim(0, len(mcnemar_df) * 1.18)
axC.set_ylim(-0.75, 2.55)
axC.set_title(
    'McNemar exact, every pair of the ten\nSmallest Holm-adjusted p-value: %.2f'
    % mcnemar_df['p (Holm)'].min(), fontsize=14, fontweight='bold', color=INK, loc='left'
)
axC.tick_params(labelsize=11.5)
axC.set_xticks([])
for s in ('top', 'right', 'bottom', 'left'):
    axC.spines[s].set_visible(False)
axC.text(
    0, -0.06,
    'Powered against the observed effect, separating the top two models at 80%%\n'
    'would need about %s held-out employees. The whole dataset is 1,470.'
    % format(int(round(need_n)), ','), fontsize=11, color=INK, va='top',
    transform=axC.transAxes, linespacing=1.5
)

# --- panel D: committed split percentile
axD = fig.add_subplot(gs[2, 1])
pct_order = cv_df.sort_values('percentile of 1234')
axD.hlines(np.arange(len(pct_order)), 0, pct_order['percentile of 1234'],
           color=GREY, linewidth=1.6, alpha=0.7)
point_colors = [
    RED if p >= 90 or p <= 10 else BLUE for p in pct_order['percentile of 1234']
]
axD.scatter(pct_order['percentile of 1234'], np.arange(len(pct_order)),
            s=95, color=point_colors, zorder=3, edgecolor='white', linewidth=1.1)
axD.axvline(50, color=INK, linestyle=':', linewidth=1.4)
axD.set_yticks(np.arange(len(pct_order)))
axD.set_yticklabels(pct_order['model'], fontsize=9.5)
axD.set_xlim(0, 100); axD.set_xlabel(
    'percentile of the committed split within its own 50 folds', fontsize=11
)
axD.set_title(
    'What seed 1234 bought each model\nRed: drew in the top or bottom tenth of its '
    'own distribution', fontsize=14, fontweight='bold', color=INK, loc='left'
)
axD.grid(axis='x', alpha=0.25); axD.set_axisbelow(True)
for s in ('top', 'right', 'left'):
    axD.spines[s].set_visible(False)

fig.savefig(OUT_PNG, dpi=140, facecolor='white', bbox_inches='tight')
plt.close(fig)
print('written: %s (%.0f KB)' % (OUT_PNG, os.path.getsize(OUT_PNG) / 1024))

written: ../img/statistical_rigour_summary.png (337 KB)


---

### Reproducing this

```
python -m nbconvert --to notebook --execute --inplace \
       --ExecutePreprocessor.timeout=3600 code/statistical_rigour.ipynb
```

Every random process in this notebook is seeded from `SEED = 1234`: the split, the two
searches, the 5,000 bootstrap draws, the 50 cross-validation partitions, and the 50
permutation shuffles. The three asserts near the top will fail loudly rather than
silently if the pipeline stops reproducing the committed split, which is the only
condition under which any comparison to the published results table would be meaningless.